# 🛡️ Animal Aggression Detection — v2.0

**Pipeline de détection d'agression animale sur flux vidéo — Architecture two-stage optimisée**

---

## Architecture complète

```
Chaque frame
    │
    ▼
[STAGE 0] YOLOv8n sentinelle (~2ms)   ← NOUVEAU v2 : filtre 78% des frames
    │  person ET dog détectés ?
    ├── NON → écrire frame brut → frame suivant
    └── OUI ↓
    │
    ▼
[STAGE 1] YOLOv8l + ByteTrack + YOLOv8l-pose (~20ms)
    │  bboxes + track_ids + keypoints
    │
    ▼
[STAGE 2] RiskScorerV2                ← NOUVEAU v2 : edge-to-edge + EMA + wrist→dog
    │  score EMA par paire (person_id, dog_id)
    │  EMA >= RISK_THRESHOLD pendant N frames ?
    ├── NON → annoter frame → frame suivant
    └── OUI ↓
    │
    ▼
[STAGE 3] LLMWorker (thread asynchrone) ← NOUVEAU v2 : non-bloquant
    │  Gemini 1.5 Flash via API Google AI (gratuit)
    │  LLM confirme ?
    ├── NON → réduire EMA → continuer
    └── OUI ↓
    │
    ▼
[STAGE 4] Agent LangGraph             ← NOUVEAU v2 : routage conditionnel
    │  evaluer_gravite → stocker_incident → generer_rapport
    │                                    → notifier_email  (serious+)
    │                                    → contacter_urgences (critical)
    └── END
```

## Corrections v2 vs v1

| # | Problème corrigé | Impact |
|---|-----------------|--------|
| P1 | Filtre sentinelle YOLOv8n | −78% GPU inutile |
| P2 | Distance edge-to-edge | Proximité précise |
| P3 | Vecteur poignet→chien | Supprime faux positifs |
| P4 | Proximité multiplicative | Score cohérent |
| P5 | Lissage EMA | Résistance aux frames bruités |
| P6 | LLM asynchrone | 0 blocage boucle principale |
| P7 | Agent LangGraph | Décision structurée |
| P8 | GPU fuse+FP16+no_grad | +30% débit |
| P9 | Schéma SQLite complet | Traçabilité totale |
| P10 | Gemini Flash remplace Claude | Stack Google AI (gratuite) |

In [ ]:
# ─── CELLULE 02 : Installation des dépendances ───────────────────────────────
# Versions fixées pour garantir la reproductibilité sur Colab T4.
# Relancer si le runtime est réinitialisé.

!pip install -q \
    "ultralytics>=8.0.0" \
    "google-generativeai>=0.7.0" \
    "langgraph>=0.2.0" \
    "langchain-google-genai>=1.0.0" \
    "langchain-core>=0.1.0" \
    "opencv-python-headless" \
    "scipy" \
    "supervision"

print("✅ Dépendances installées")
print("   ultralytics (YOLOv8 + ByteTrack)")
print("   google-generativeai (Gemini 1.5 Flash)")
print("   langgraph   (agent décisionnel)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 20.8 MB/s eta 0:00:00
✅ Dépendances installées
   ultralytics (YOLOv8 + ByteTrack)
   google-generativeai (Gemini 1.5 Flash)
   langgraph   (agent décisionnel)


In [ ]:
# ─── CELLULE 03 : Vérification environnement + imports ───────────────────────
# Vérifie GPU, importe toutes les bibliothèques et configure le logging.
# Si un import échoue, le message indique quelle cellule relancer.

import sys, os, logging, json, re, base64, threading, queue
import uuid, time, smtplib, csv
from datetime import datetime
from collections import defaultdict, deque
from pathlib import Path
from math import sqrt as _sqrt
from typing import Optional

import numpy as np
import cv2
import torch

# ── Vérification GPU ──────────────────────────────────────────────────────────
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU : {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️  CPU uniquement — inférence plus lente (~10× vs T4)")

# ── YOLO / ByteTrack ──────────────────────────────────────────────────────────
from ultralytics import YOLO
print("✅ ultralytics importé")

# ── Google Gemini ──────────────────────────────────────────────────────────────
try:
    import google.generativeai as genai
    print("✅ google-generativeai importé")
except ImportError:
    print("❌ google-generativeai manquant — relancer cellule 02")

# ── LangGraph ────────────────────────────────────────────────────────────────
try:
    from langgraph.graph import StateGraph, END
    from typing_extensions import TypedDict
    LANGGRAPH_OK = True
    print("✅ langgraph importé")
except ImportError:
    LANGGRAPH_OK = False
    # TypedDict standard comme fallback
    from typing import TypedDict
    print("⚠️  langgraph manquant — pip install langgraph>=0.2.0 (cellule 02)")

# ── Email ─────────────────────────────────────────────────────────────────────
from email.mime.multipart import MIMEMultipart
from email.mime.text      import MIMEText
from email.mime.image     import MIMEImage

print(f"\n✅ Environnement prêt — device={DEVICE}, Python {sys.version.split()[0]}")

✅ GPU : Tesla T4 (15.6 GB VRAM)
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ ultralytics importé


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ google-generativeai importé
✅ langgraph importé

✅ Environnement prêt — device=cuda, Python 3.12.13


In [ ]:
# ─── CELLULE 04 : Configuration globale ──────────────────────────────────────
# Point unique de configuration. Aucune valeur sensible hardcodée.
# Utilise google.colab.userdata avec fallback os.environ.
# Les clés manquantes lèvent ValueError au moment de l'usage, pas ici.

try:
    from google.colab import userdata as _ud
    def _get(key): return _ud.get(key)
except Exception:
    def _get(key): return None

# ── Clés API (récupérées à l'usage par LLMWorker) ────────────────────────────
GEMINI_API_KEY = _get('GEMINI_API_KEY') or os.environ.get('GEMINI_API_KEY', '')

# ── Modèles ───────────────────────────────────────────────────────────────────
SENTINEL_MODEL = "yolov8n.pt"      # Léger : filtre les frames sans person+dog
DETECT_MODEL   = "yolov8l.pt"      # Lourd : détection précise
POSE_MODEL     = "yolov8l-pose.pt" # Pose estimation 17 keypoints
LLM_MODEL      = "gemini-1.5-flash"
USE_FP16       = (DEVICE == 'cuda') # FP16 réduit VRAM de ~40% sur GPU

# ── Classes COCO ──────────────────────────────────────────────────────────────
PERSON_CLASS_ID = 0   # 'person'
DOG_CLASS_ID    = 16  # 'dog'

# ── Indices keypoints COCO (17 points) ───────────────────────────────────────
# 0=nez  1=oeil_g  2=oeil_d  3=oreille_g  4=oreille_d
# 5=ep_g  6=ep_d  7=coude_g  8=coude_d  9=poig_g  10=poig_d
# 11=hanche_g  12=hanche_d  13=genou_g  14=genou_d  15=chev_g  16=chev_d
KP_SHOULDER_L, KP_SHOULDER_R = 5, 6
KP_ELBOW_L,    KP_ELBOW_R    = 7, 8
KP_WRIST_L,    KP_WRIST_R    = 9, 10
KP_HIP_L,      KP_HIP_R      = 11, 12
KP_KNEE_L,     KP_KNEE_R     = 13, 14
KP_ANKLE_L,    KP_ANKLE_R    = 15, 16

# ── Seuils détection ──────────────────────────────────────────────────────────
SENTINEL_CONF = 0.35  # Bas pour ne rater aucun candidat (sentinelle)
DETECT_CONF   = 0.40
POSE_CONF     = 0.40

# ── RiskScorerV2 ──────────────────────────────────────────────────────────────
PROXIMITY_THRESHOLD_PX = 120  # Distance edge-to-edge critique (pixels)
EMA_ALPHA              = 0.35 # α=0.35 : fenêtre effective ≈3 frames, résistant au bruit
MIN_SUSTAINED_FRAMES   = 4    # Frames EMA≥seuil avant de solliciter le LLM
RISK_THRESHOLD         = 0.50 # Seuil EMA déclenchant la surveillance renforcée

# ── Anti-spam LLM ─────────────────────────────────────────────────────────────
MIN_FRAMES_BETWEEN_LLM = 375  # ≈15s à 25fps — évite les appels répétés pour le même incident

# ── Niveaux de sévérité (utilisés par l'agent LangGraph) ─────────────────────
SEVERITY_THRESHOLDS = {
    'minor':    0.45,
    'serious':  0.65,
    'critical': 0.82
}

# ── Stockage (configurable via env, défaut Colab /content/, fallback ./aggression_data) ─
_DEFAULT_BASE = "/content" if os.path.exists("/content") else "./aggression_data"
BASE_DIR     = os.environ.get("AGGRESSION_BASE_DIR", _DEFAULT_BASE)
DB_PATH      = os.path.join(BASE_DIR, "aggression_events.db")
CAPTURES_DIR = os.path.join(BASE_DIR, "captures")
REPORTS_DIR  = os.path.join(BASE_DIR, "captures", "reports")
os.makedirs(CAPTURES_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR,  exist_ok=True)

# Backend integration — set to forward confirmed incidents to the production agent.
BACKEND_URL  = os.environ.get("PETADVISOR_BACKEND_URL", "")  # e.g. http://localhost:8000

# ── Notifications ─────────────────────────────────────────────────────────────
# Mettre EMAIL_ENABLED=True et renseigner les champs via userdata Colab
EMAIL_ENABLED    = True
SMTP_HOST        = "smtp.gmail.com"
SMTP_PORT        = 587
SMTP_USER        = _get('SMTP_USER')        or ""
SMTP_PASS        = _get('SMTP_PASS')        or ""
ALERT_RECIPIENTS = ["ferchichi.adham00@gmail.com"]    # ex: ["superviseur@example.com"]
SPA_EMAIL        = "adhamferchihci3@gmail.com"    # Email SPA pour maltraitance animale
MEDICAL_EMAIL    = "khalidfirchichi@gmail.com"    # Email urgences médicales (morsures)

# ── Logging structuré ─────────────────────────────────────────────────────────
DEBUG_MODE = False
logging.basicConfig(
    level=logging.DEBUG if DEBUG_MODE else logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("aggression_v2")

print("✅ Configuration chargée")
print(f"   LLM      : {LLM_MODEL}")
print(f"   FP16     : {USE_FP16}")
print(f"   Seuil    : {RISK_THRESHOLD}  |  α EMA : {EMA_ALPHA}")
print(f"   Captures : {CAPTURES_DIR}")
print(f"   DB       : {DB_PATH}")
if not GEMINI_API_KEY:
    print("   ⚠️  GEMINI_API_KEY absente — stage LLM bypassé")

SecretNotFoundError: Secret SMTP_USER does not exist.

In [ ]:
# ─── CELLULE 05 : Chargement des modèles + optimisations GPU ─────────────────
# Optimisations v2 (P8) : model.fuse() + FP16 + torch.no_grad() dans pipeline.
# Trois modèles : sentinelle légère, détection lourde, pose estimation.

def _load_and_optimize(model_path: str, fp16: bool = False) -> YOLO:
    """
    Charge un modèle YOLO et applique les optimisations GPU.

    Args:
        model_path: Chemin ou nom du fichier .pt (téléchargé automatiquement).
        fp16: Active FP16 si True et DEVICE=='cuda'.
    Returns:
        Modèle YOLO optimisé, prêt pour l'inférence.
    """
    model = YOLO(model_path)
    model.to(DEVICE)
    model.fuse()       # Fusion Conv+BatchNorm → réduit VRAM d'environ 20%
    if fp16 and DEVICE == 'cuda':
        model.half()   # Passage en demi-précision → +30% débit sur T4
    return model

logger.info("Chargement des modèles YOLO...")

# Sentinelle : nano, très rapide, tourne sur 100% des frames (~2ms/frame GPU)
model_sentinel = _load_and_optimize(SENTINEL_MODEL, USE_FP16)
logger.info(f"✅ Sentinelle : {SENTINEL_MODEL}")

# Détection : large, précis, activé seulement si sentinelle détecte person+dog
model_detect = _load_and_optimize(DETECT_MODEL, USE_FP16)
logger.info(f"✅ Détection  : {DETECT_MODEL}")

# Pose : large, activé seulement après sentinelle pour les keypoints humains
model_pose = _load_and_optimize(POSE_MODEL, USE_FP16)
logger.info(f"✅ Pose       : {POSE_MODEL}")

# Purge du cache GPU après chargement pour libérer les buffers temporaires
if DEVICE == 'cuda':
    torch.cuda.empty_cache()
    used_gb = torch.cuda.memory_allocated() / 1e9
    print(f"   VRAM utilisée après chargement : {used_gb:.2f} GB")

print("\n✅ Modèles prêts (sentinel + detect + pose)")
print(f"   FP16 actif : {USE_FP16}")
print("   P8 corrigé : fuse() + FP16 + torch.no_grad() dans pipeline")

In [ ]:
# ─── CELLULE 06 : Base de données SQLite — Schéma v2 ─────────────────────────
# Schéma enrichi (P9) : tous les champs de IncidentState + actions agent.
# Les champs list/dict sont sérialisés en JSON string pour SQLite.

import sqlite3, csv

# Schéma complet — ne pas modifier manuellement (migration auto non prévue)
SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS aggression_events (
    id                  INTEGER PRIMARY KEY AUTOINCREMENT,
    incident_id         TEXT    NOT NULL,
    timestamp           TEXT    NOT NULL,
    video_source        TEXT,
    frame_number        INTEGER,

    incident_type       TEXT,           -- human_to_dog | dog_to_human
    person_track_id     INTEGER,
    dog_track_id        INTEGER,
    distance_px         REAL,           -- edge-to-edge en pixels

    ema_score           REAL,           -- valeur EMA au moment de l'alerte
    ema_history         TEXT,           -- JSON array des 8 dernières valeurs EMA
    sustained_frames    INTEGER,        -- frames consécutives >= RISK_THRESHOLD
    risk_scores_json    TEXT,           -- JSON dict complet de compute_pair()
    evidence_list       TEXT,           -- JSON array des signaux détectés (>0.5)

    llm_confirmed       INTEGER DEFAULT 0,
    llm_confidence      REAL,
    llm_model           TEXT,
    llm_reason          TEXT,
    llm_severity        TEXT,           -- minor | serious | critical
    llm_recommended     TEXT,           -- report_only | notify_email | ...

    severity            TEXT,           -- copié depuis llm_severity par l'agent
    report_path         TEXT,
    capture_path        TEXT,
    notifications_sent  TEXT,           -- JSON array (email, spa, medical...)
    actions_taken       TEXT,           -- JSON array des actions effectuées
    processing_time_ms  REAL,           -- temps total de traitement

    notified            INTEGER DEFAULT 0,
    created_at          TEXT    DEFAULT CURRENT_TIMESTAMP
)
"""

def init_database(db_path: str) -> None:
    """Crée la base et le schéma v2. Idempotent (IF NOT EXISTS)."""
    conn = sqlite3.connect(db_path)
    conn.executescript(SCHEMA_SQL)
    conn.commit()
    conn.close()
    logger.info(f"Base initialisée : {db_path}")

def save_event(db_path: str, state: dict) -> int:
    """
    Insère un IncidentState complet en base.
    Les champs list/dict sont automatiquement sérialisés en JSON.

    Args:
        db_path: Chemin SQLite.
        state: Dict compatible IncidentState.
    Returns:
        ID auto-incrémenté de la ligne insérée.
    """
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()
    cur.execute("""
        INSERT INTO aggression_events (
            incident_id, timestamp, video_source, frame_number,
            incident_type, person_track_id, dog_track_id, distance_px,
            ema_score, ema_history, sustained_frames, risk_scores_json, evidence_list,
            llm_confirmed, llm_confidence, llm_model, llm_reason, llm_severity, llm_recommended,
            severity, report_path, capture_path, notifications_sent, actions_taken,
            processing_time_ms, notified
        ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        state.get('incident_id',''),       state.get('timestamp',''),
        state.get('video_source',''),      state.get('frame_number',-1),
        state.get('incident_type',''),     state.get('person_track_id',-1),
        state.get('dog_track_id',-1),      state.get('distance_px',0.0),
        state.get('ema_score',0.0),
        json.dumps(list(state.get('ema_history',[]))),
        state.get('sustained_frames',0),
        json.dumps(state.get('risk_scores',{})),
        json.dumps(state.get('evidence_list',[])),
        1 if state.get('llm_confirmed') else 0,
        state.get('llm_confidence',0.0),   state.get('llm_model', LLM_MODEL),
        state.get('llm_reason',''),        state.get('llm_severity',''),
        state.get('llm_recommended_action',''),
        state.get('severity',''),          state.get('report_path',''),
        state.get('capture_path',''),
        json.dumps(state.get('notifications_sent',[])),
        json.dumps(state.get('actions_taken',[])),
        state.get('processing_time_ms',0.0),
        1 if state.get('notifications_sent') else 0
    ))
    event_id = cur.lastrowid
    conn.commit()
    conn.close()
    return event_id

def get_incident_by_id(incident_id: str) -> Optional[dict]:
    """Retourne un incident par UUID ou None si inexistant."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    row  = conn.execute(
        "SELECT * FROM aggression_events WHERE incident_id=?", (incident_id,)
    ).fetchone()
    conn.close()
    return dict(row) if row else None

def get_incidents_by_severity(severity: str, limit: int = 20) -> list:
    """Retourne les N derniers incidents pour un niveau de sévérité donné."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT * FROM aggression_events WHERE severity=? ORDER BY id DESC LIMIT ?",
        (severity, limit)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def get_incidents_by_type(incident_type: str, limit: int = 20) -> list:
    """Retourne les N derniers incidents pour un type donné (human_to_dog / dog_to_human)."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute(
        "SELECT * FROM aggression_events WHERE incident_type=? ORDER BY id DESC LIMIT ?",
        (incident_type, limit)
    ).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def get_stats_summary() -> dict:
    """Retourne les statistiques globales : total, confirmés, par type et sévérité."""
    conn = sqlite3.connect(DB_PATH)
    cur  = conn.cursor()
    total     = cur.execute("SELECT COUNT(*) FROM aggression_events").fetchone()[0]
    confirmed = cur.execute(
        "SELECT COUNT(*) FROM aggression_events WHERE llm_confirmed=1"
    ).fetchone()[0]
    by_type = dict(cur.execute(
        "SELECT incident_type, COUNT(*) FROM aggression_events GROUP BY incident_type"
    ).fetchall())
    by_sev = dict(cur.execute(
        "SELECT severity, COUNT(*) FROM aggression_events GROUP BY severity"
    ).fetchall())
    conn.close()
    return {
        'total': total, 'confirmed': confirmed,
        'false_positives': total - confirmed,
        'by_type': by_type, 'by_severity': by_sev
    }

def export_to_csv(output_path: str) -> None:
    """Exporte tous les incidents vers un fichier CSV."""
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    rows = conn.execute("SELECT * FROM aggression_events ORDER BY id").fetchall()
    conn.close()
    if not rows:
        logger.warning("Aucun incident à exporter")
        return
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows([dict(r) for r in rows])
    logger.info(f"Export CSV : {output_path} ({len(rows)} lignes)")

# Initialisation au chargement de la cellule
init_database(DB_PATH)
print("✅ Module base de données prêt (schéma v2 — P9 corrigé)")
print(f"   Colonnes : incident_type, severity, ema_history, evidence_list, actions_taken")

In [ ]:
# ─── CELLULE 07 : RiskScorerV2 ───────────────────────────────────────────────
# Corrections P2 (edge-to-edge), P3 (wrist→dog), P4 (multiplicatif), P5 (EMA)
# 7 méthodes : A) distance  B) proximity  C) wrist_dir  D) H→D  E) D→H  F) EMA  G) compute

class RiskScorerV2:
    """
    Calcule les scores de risque d'agression pour une paire (personne, chien).

    Différences majeures vs v1 :
    - Distance edge-to-edge au lieu de centre-à-centre (P2)
    - Composante directionnelle poignet→chien (P3)
    - Score final = geste × facteur_proximité (multiplicatif, P4)
    - Lissage EMA par paire au lieu d'un compteur binaire (P5)
    """

    def __init__(self, history_len: int = 8):
        # Historiques indexés par track_id — deque O(1) en append/pop
        self._kp_history: dict        = defaultdict(lambda: deque(maxlen=history_len))
        self._bbox_history: dict      = defaultdict(lambda: deque(maxlen=history_len))
        self._dog_ratio_history: dict = defaultdict(lambda: deque(maxlen=8))
        # Scores EMA par clé de paire ('h2d'|'d2h', pid, did)
        self.ema_scores: dict  = {}
        # 8 dernières valeurs EMA par paire pour les rapports
        self.ema_history: dict = defaultdict(lambda: deque(maxlen=8))
        self.history_len = history_len

    # ── A) Distance edge-to-edge ───────────────────────────────────────────────

    def _edge_to_edge_distance(self, bbox1: np.ndarray, bbox2: np.ndarray) -> float:
        """
        Distance minimale entre les bords des deux rectangles (P2 corrigé).
        Retourne 0.0 si les bboxes se chevauchent ou se touchent.

        Args:
            bbox1, bbox2: [x1, y1, x2, y2]
        Returns:
            Distance en pixels (float >= 0).
        """
        # Écart horizontal entre les bords les plus proches
        dist_x = max(0.0, max(bbox1[0], bbox2[0]) - min(bbox1[2], bbox2[2]))
        # Écart vertical entre les bords les plus proches
        dist_y = max(0.0, max(bbox1[1], bbox2[1]) - min(bbox1[3], bbox2[3]))
        return _sqrt(dist_x**2 + dist_y**2)

    # ── B) Facteur de proximité multiplicatif ──────────────────────────────────

    def _proximity_factor(self, distance_px: float) -> float:
        """
        Multiplicateur [0–1] basé sur la distance edge-to-edge (P4 corrigé).
        Ce facteur MULTIPLIE le score geste — pas d'addition.
        Un geste agressif à 400px du chien donne un score quasi nul.

        Args:
            distance_px: Distance edge-to-edge (méthode A).
        Returns:
            1.0 si contact, 0.0 si distance >= 2×PROXIMITY_THRESHOLD_PX.
        """
        if distance_px == 0.0:
            return 1.0  # Chevauchement = contact direct
        threshold_x2 = PROXIMITY_THRESHOLD_PX * 2.0
        if distance_px >= threshold_x2:
            return 0.0
        return 1.0 - (distance_px / threshold_x2)

    # ── C) Score directionnel poignet → cible ─────────────────────────────────

    def _wrist_toward_target_score(
        self, person_kps: np.ndarray, target_bbox: np.ndarray, person_track_id: int
    ) -> float:
        """
        Vérifie si le mouvement du poignet est orienté vers la cible (P3 corrigé).
        Signal le plus discriminant : un bras levé qui ne pointe pas vers le chien
        ne devrait pas scorer.

        Méthode :
          1. Vecteur mouvement poignet = pos_actuelle − pos_précédente (historique)
          2. Vecteur poignet → centre_cible
          3. cosinus(v_mvt, v_cible) clampé à [0, 1]

        Args:
            person_kps: Keypoints [17, 3] du frame actuel.
            target_bbox: Bbox [x1,y1,x2,y2] du chien.
            person_track_id: Pour accéder à l'historique de keypoints.
        Returns:
            Score directionnel [0.0–1.0]. 0 si historique insuffisant.
        """
        kp_hist = self._kp_history[person_track_id]
        if len(kp_hist) < 2:
            return 0.0  # Pas encore d'historique pour calculer un vecteur de mouvement

        prev_kps = kp_hist[-2]  # Frame précédent
        # Centre de la cible (chien)
        target_center = np.array([
            (target_bbox[0] + target_bbox[2]) / 2.0,
            (target_bbox[1] + target_bbox[3]) / 2.0
        ])
        best = 0.0

        for w_idx in [KP_WRIST_L, KP_WRIST_R]:
            if w_idx >= len(person_kps) or w_idx >= len(prev_kps):
                continue
            if person_kps[w_idx][2] < 0.3 or prev_kps[w_idx][2] < 0.3:
                continue  # Keypoint peu fiable (conf < 30%)

            curr_pos   = person_kps[w_idx][:2]
            prev_pos   = prev_kps[w_idx][:2]
            v_movement = curr_pos - prev_pos          # Vecteur déplacement poignet
            v_toward   = target_center - curr_pos     # Vecteur vers le chien

            norm_m = np.linalg.norm(v_movement)
            norm_t = np.linalg.norm(v_toward)
            if norm_m < 1.0 or norm_t < 1.0:
                continue  # Mouvement ou distance négligeable

            cosine = np.dot(v_movement, v_toward) / (norm_m * norm_t)
            best   = max(best, max(0.0, cosine))  # Clamp à 0 si poignet s'éloigne

        return best

    # ── D) Score geste H→D ────────────────────────────────────────────────────

    def _human_to_dog_gesture_score(
        self, track_id: int, keypoints: np.ndarray, dog_bbox: np.ndarray
    ) -> dict:
        """
        Score de geste agressif humain→chien (4 composantes pondérées).
        La composante wrist_toward_dog (poids 0.40) est le signal le plus discriminant.

        Args:
            track_id: ID de la personne (pour historiques).
            keypoints: Array [17, 3].
            dog_bbox: Bbox [x1,y1,x2,y2] du chien.
        Returns:
            Dict avec raw_gesture_score, détails des composantes, evidence list.
        """
        evidence = []

        def valid(idx):
            return idx < len(keypoints) and keypoints[idx][2] > 0.3

        # Composante 1 : Poignet au-dessus de l'épaule — bras levé (poids 0.20)
        # En image : y diminue vers le haut, donc épaule_y - poignet_y > 0 = bras levé
        wrist_above = 0.0
        for w_idx, s_idx in [(KP_WRIST_L, KP_SHOULDER_L), (KP_WRIST_R, KP_SHOULDER_R)]:
            if valid(w_idx) and valid(s_idx):
                dy = keypoints[s_idx][1] - keypoints[w_idx][1]
                wrist_above = max(wrist_above, min(1.0, max(0.0, dy / 80.0)))
        if wrist_above > 0.5:
            evidence.append(f"bras_leve:{wrist_above:.2f}")

        # Composante 2 : Extension du bras (poids 0.20)
        # direct_len/total_len : 1.0=totalement tendu, 0.5=coude à 90°
        arm_ext = 0.0
        for s_idx, e_idx, w_idx in [
            (KP_SHOULDER_L, KP_ELBOW_L, KP_WRIST_L),
            (KP_SHOULDER_R, KP_ELBOW_R, KP_WRIST_R)
        ]:
            if valid(s_idx) and valid(e_idx) and valid(w_idx):
                total  = (np.linalg.norm(keypoints[e_idx][:2] - keypoints[s_idx][:2]) +
                          np.linalg.norm(keypoints[w_idx][:2] - keypoints[e_idx][:2]))
                direct = np.linalg.norm(keypoints[w_idx][:2] - keypoints[s_idx][:2])
                if total > 5.0:
                    arm_ext = max(arm_ext, direct / total)
        if arm_ext > 0.5:
            evidence.append(f"bras_tendu:{arm_ext:.2f}")

        # Composante 3 : Direction poignet→chien (poids 0.40) — signal le plus fort
        wrist_dir = self._wrist_toward_target_score(keypoints, dog_bbox, track_id)
        if wrist_dir > 0.5:
            evidence.append(f"poignet_vers_chien:{wrist_dir:.2f}")

        # Composante 4 : Vélocité des membres supérieurs (poids 0.20)
        limb_vel = self._get_limb_velocity(track_id)
        if limb_vel > 0.5:
            evidence.append(f"vitesse_membres:{limb_vel:.2f}")

        raw = wrist_above*0.20 + arm_ext*0.20 + wrist_dir*0.40 + limb_vel*0.20
        return {
            'raw_gesture_score':    round(raw, 3),
            'wrist_above_shoulder': round(wrist_above, 3),
            'arm_extension':        round(arm_ext, 3),
            'wrist_toward_dog':     round(wrist_dir, 3),
            'limb_velocity':        round(limb_vel, 3),
            'evidence':             evidence
        }

    # ── E) Score agression D→H ────────────────────────────────────────────────

    def _dog_to_human_aggression_score(
        self, dog_track_id: int, dog_bbox: np.ndarray,
        person_track_id: int, person_bbox: np.ndarray
    ) -> dict:
        """
        Score d'agression chien→humain (3 composantes pondérées).

        Args:
            dog_track_id, dog_bbox: Identité et position du chien.
            person_track_id, person_bbox: Identité et position de la personne.
        Returns:
            Dict avec raw_aggression_score, détails, evidence list.
        """
        evidence = []

        # Composante 1 : Approche directionnelle du chien (poids 0.50)
        # cosinus(mouvement_chien, vecteur_chien→personne) × vélocité_chien
        dog_vel  = self._get_entity_velocity(dog_track_id)
        dog_hist = self._bbox_history[dog_track_id]
        dog_approach = 0.0
        if dog_vel > 0.05 and len(dog_hist) >= 2:
            dh = list(dog_hist)
            d_prev = np.array([(dh[-2][0]+dh[-2][2])/2, (dh[-2][1]+dh[-2][3])/2])
            d_curr = np.array([(dh[-1][0]+dh[-1][2])/2, (dh[-1][1]+dh[-1][3])/2])
            p_center = np.array([(person_bbox[0]+person_bbox[2])/2,
                                  (person_bbox[1]+person_bbox[3])/2])
            v_mvt    = d_curr - d_prev
            v_toward = p_center - d_curr
            nm, nt = np.linalg.norm(v_mvt), np.linalg.norm(v_toward)
            if nm > 1.0 and nt > 1.0:
                cos = np.dot(v_mvt, v_toward) / (nm * nt)
                dog_approach = max(0.0, cos) * dog_vel
        if dog_approach > 0.3:
            evidence.append(f"chien_fonce:{dog_approach:.2f}")

        # Composante 2 : Changement de ratio h/w bbox chien (poids 0.20)
        # Détecte les changements de posture : accroupissement (attaque), saut
        h = dog_bbox[3] - dog_bbox[1]
        w = dog_bbox[2] - dog_bbox[0]
        ratio_curr = h / max(1.0, w)
        ratio_hist = self._dog_ratio_history[dog_track_id]
        bbox_ratio = 0.0
        if len(ratio_hist) >= 3:
            ratio_mean = float(np.mean(list(ratio_hist)))
            delta = abs(ratio_curr - ratio_mean) / max(0.01, ratio_mean)
            bbox_ratio = min(1.0, delta * 3.0)
        ratio_hist.append(ratio_curr)
        if bbox_ratio > 0.3:
            evidence.append(f"posture_agressive:{bbox_ratio:.2f}")

        # Composante 3 : Fuite de la personne (poids 0.30)
        # cosinus(mouvement_personne, opposé_chien→personne) × vélocité_personne
        p_vel  = self._get_entity_velocity(person_track_id)
        p_hist = self._bbox_history[person_track_id]
        human_flee = 0.0
        if p_vel > 0.05 and len(p_hist) >= 2:
            ph = list(p_hist)
            p_prev = np.array([(ph[-2][0]+ph[-2][2])/2, (ph[-2][1]+ph[-2][3])/2])
            p_curr = np.array([(ph[-1][0]+ph[-1][2])/2, (ph[-1][1]+ph[-1][3])/2])
            d_center = np.array([(dog_bbox[0]+dog_bbox[2])/2, (dog_bbox[1]+dog_bbox[3])/2])
            v_flee   = p_curr - p_prev
            v_opp    = -(d_center - p_curr)  # Direction opposée au chien
            nf, no   = np.linalg.norm(v_flee), np.linalg.norm(v_opp)
            if nf > 1.0 and no > 1.0:
                cos = np.dot(v_flee, v_opp) / (nf * no)
                human_flee = max(0.0, cos) * p_vel
        if human_flee > 0.3:
            evidence.append(f"personne_fuit:{human_flee:.2f}")

        raw = dog_approach*0.50 + bbox_ratio*0.20 + human_flee*0.30
        return {
            'raw_aggression_score':  round(raw, 3),
            'dog_approach_score':    round(dog_approach, 3),
            'dog_bbox_ratio_change': round(bbox_ratio, 3),
            'human_fleeing_score':   round(human_flee, 3),
            'evidence':              evidence
        }

    # ── F) Mise à jour EMA ────────────────────────────────────────────────────

    def _update_ema(self, track_key: tuple, new_score: float) -> float:
        """
        Met à jour l'EMA pour une paire donnée (P5 corrigé).
        EMA_t = α × score_t + (1-α) × EMA_{t-1}
        Avec α=0.35 : fenêtre effective ≈3 frames, résistant aux frames bruités.

        Args:
            track_key: ('h2d'|'d2h', person_id, dog_id)
            new_score: Score brut du frame courant [0, 1].
        Returns:
            Valeur EMA lissée [0, 1].
        """
        if track_key not in self.ema_scores:
            self.ema_scores[track_key] = 0.0
        ema = EMA_ALPHA * new_score + (1.0 - EMA_ALPHA) * self.ema_scores[track_key]
        self.ema_scores[track_key] = ema
        self.ema_history[track_key].append(round(ema, 3))
        return ema

    # ── Utilitaires internes ───────────────────────────────────────────────────

    def _get_entity_velocity(self, track_id: int) -> float:
        """Vélocité normalisée [0,1] basée sur le déplacement de la bbox centre."""
        hist = list(self._bbox_history[track_id])
        if len(hist) < 2:
            return 0.0
        velocities = []
        for i in range(1, len(hist)):
            pc = np.array([(hist[i-1][0]+hist[i-1][2])/2, (hist[i-1][1]+hist[i-1][3])/2])
            cc = np.array([(hist[i][0]+hist[i][2])/2,   (hist[i][1]+hist[i][3])/2])
            velocities.append(np.linalg.norm(cc - pc))
        return min(1.0, float(np.mean(velocities)) / 60.0)

    def _get_limb_velocity(self, track_id: int) -> float:
        """Vélocité normalisée [0,1] des coudes et poignets."""
        hist = list(self._kp_history[track_id])
        if len(hist) < 2:
            return 0.0
        arm_idx = [KP_ELBOW_L, KP_ELBOW_R, KP_WRIST_L, KP_WRIST_R]
        vels    = []
        for i in range(1, len(hist)):
            for idx in arm_idx:
                if (idx < len(hist[i-1]) and idx < len(hist[i]) and
                        hist[i-1][idx][2] > 0.3 and hist[i][idx][2] > 0.3):
                    vels.append(np.linalg.norm(hist[i][idx][:2] - hist[i-1][idx][:2]))
        return min(1.0, float(np.mean(vels)) / 60.0) if vels else 0.0

    # ── G) Point d'entrée principal ───────────────────────────────────────────

    def compute_pair(
        self,
        person_track_id: int,
        person_kps:      Optional[np.ndarray],
        person_bbox:     np.ndarray,
        dog_track_id:    int,
        dog_bbox:        np.ndarray
    ) -> dict:
        """
        Calcule tous les scores pour une paire (personne, chien) et met à jour les EMA.

        Args:
            person_track_id: ID ByteTrack de la personne.
            person_kps: Keypoints [17,3] ou None si pose non disponible.
            person_bbox: [x1,y1,x2,y2].
            dog_track_id: ID ByteTrack du chien.
            dog_bbox: [x1,y1,x2,y2].
        Returns:
            Dict complet : human_to_dog, dog_to_human, dominant_type, EMA, distance.
        """
        # Mise à jour des historiques de position
        self._bbox_history[person_track_id].append(person_bbox.copy())
        self._bbox_history[dog_track_id].append(dog_bbox.copy())
        if person_kps is not None:
            self._kp_history[person_track_id].append(person_kps.copy())

        # Distance edge-to-edge et facteur de proximité multiplicatif
        dist_px     = self._edge_to_edge_distance(person_bbox, dog_bbox)
        prox_factor = self._proximity_factor(dist_px)

        tk = (person_track_id, dog_track_id)

        # ── Score H→D : geste × proximité ────────────────────────────────────
        h2d_raw = {'raw_gesture_score': 0.0, 'wrist_above_shoulder': 0.0,
                   'arm_extension': 0.0, 'wrist_toward_dog': 0.0,
                   'limb_velocity': 0.0, 'evidence': []}
        if person_kps is not None:
            h2d_raw = self._human_to_dog_gesture_score(person_track_id, person_kps, dog_bbox)
        h2d_final = h2d_raw['raw_gesture_score'] * prox_factor
        h2d_ema   = self._update_ema(('h2d',) + tk, h2d_final)

        # ── Score D→H : approche × proximité ─────────────────────────────────
        d2h_raw   = self._dog_to_human_aggression_score(dog_track_id, dog_bbox,
                                                        person_track_id, person_bbox)
        d2h_final = d2h_raw['raw_aggression_score'] * prox_factor
        d2h_ema   = self._update_ema(('d2h',) + tk, d2h_final)

        # Type dominant = celui avec le final_score le plus élevé
        if h2d_final >= d2h_final:
            dom_type, dom_score, dom_ema = 'human_to_dog', h2d_final, h2d_ema
        else:
            dom_type, dom_score, dom_ema = 'dog_to_human', d2h_final, d2h_ema

        return {
            'human_to_dog': {
                'raw_gesture_score': h2d_raw['raw_gesture_score'],
                'ema_score':         round(h2d_ema, 3),
                'proximity_factor':  round(prox_factor, 3),
                'final_score':       round(h2d_final, 3),
                'evidence':          h2d_raw['evidence'],
                'details':           h2d_raw
            },
            'dog_to_human': {
                'raw_aggression_score': d2h_raw['raw_aggression_score'],
                'ema_score':            round(d2h_ema, 3),
                'proximity_factor':     round(prox_factor, 3),
                'final_score':          round(d2h_final, 3),
                'evidence':             d2h_raw['evidence'],
                'details':              d2h_raw
            },
            'dominant_type':    dom_type,
            'dominant_score':   round(dom_score, 3),
            'dominant_ema':     round(dom_ema, 3),
            'distance_px':      round(dist_px, 1),
            'proximity_factor': round(prox_factor, 3)
        }


risk_scorer = RiskScorerV2(history_len=8)
print("✅ RiskScorerV2 prêt")
print("   P2 corrigé : distance edge-to-edge")
print("   P3 corrigé : wrist_toward_target_score")
print("   P4 corrigé : score = geste × facteur_proximité (multiplicatif)")
print("   P5 corrigé : EMA α=0.35 par paire")

In [ ]:
# ─── CELLULE 08 : LLMWorker — Thread LLM asynchrone ──────────────────────────────
# Résout P6 : l'appel Gemini ne bloque plus la boucle principale.
# Architecture : boucle → queue_in → thread daemon → queue_out → boucle

import PIL.Image, io as _io

class LLMWorker:
    """
    Worker asynchrone pour les confirmations via Gemini (API Google AI).

    La boucle principale soumet des tâches via submit() et récupère les
    résultats de manière non-bloquante via get_result(). Le thread daemon
    s'arrête automatiquement à la fin du notebook Colab.
    """

    def __init__(self, api_key: str, model: str = LLM_MODEL):
        """
        Args:
            api_key: Clé API Google AI — ValueError si vide.
            model: ID du modèle Gemini à utiliser.
        """
        if not api_key:
            raise ValueError(
                "❌ GEMINI_API_KEY manquante. "
                "Ajoutez-la dans Colab → Secrets → GEMINI_API_KEY."
            )
        genai.configure(api_key=api_key)
        self._model_name = model
        self.model = genai.GenerativeModel(
            model_name=model,
            generation_config=genai.GenerationConfig(
                max_output_tokens=512,
                temperature=0.1,
            )
        )
        # maxsize=4 : évite l'accumulation de tâches périmées si le LLM est lent
        self.queue_in  = queue.Queue(maxsize=4)
        self.queue_out = queue.Queue(maxsize=4)
        # daemon=True : le thread s'arrête sans bloquer la fin du runtime
        self.thread = threading.Thread(target=self._worker_loop, daemon=True)
        self.thread.start()
        logger.info(f"LLMWorker démarré — modèle={model}")

    def _worker_loop(self):
        """Boucle infinie : consomme queue_in et publie dans queue_out."""
        while True:
            task = self.queue_in.get()
            if task is None:
                break  # Signal d'arrêt propre envoyé par stop()
            result = self._call_gemini(task)
            try:
                self.queue_out.put(result, timeout=2)
            except queue.Full:
                logger.warning("LLMWorker: queue_out pleine — résultat abandonné")

    def _call_gemini(self, task: dict) -> dict:
        """
        Appelle Gemini Vision avec l'image et le contexte de détection.
        2 tentatives max avec 2s d'attente entre chaque.

        Args:
            task: Dict contenant 'frame' (np.ndarray BGR), 'type' (str) et scores.
        Returns:
            Dict normalisé : confirmed, confidence, severity, reason, recommended_action.
        """
        # Conversion BGR→RGB puis PIL Image (format attendu par Gemini)
        rgb     = cv2.cvtColor(task['frame'], cv2.COLOR_BGR2RGB)
        pil_img = PIL.Image.fromarray(rgb)

        inc_type = task.get('type', 'unknown')

        system_prompt = (
            "Tu es un système expert en détection d'agression animale pour réseau "
            "de surveillance. Analyse l'image et le contexte fourni. "
            "Réponds UNIQUEMENT en JSON valide sans balise markdown."
        )

        # Prompt adapté selon le type d'incident suspect
        if inc_type == 'human_to_dog':
            ctx = (
                f"CONTEXTE DÉTECTION AUTOMATIQUE :\n"
                f"- Type suspect : HUMAIN agresse CHIEN\n"
                f"- Score EMA actuel : {task.get('ema_score',0):.3f} / 1.0\n"
                f"- Score geste : {task.get('gesture_score',0):.3f} / 1.0\n"
                f"- Facteur proximité : {task.get('proximity_factor',0):.3f} / 1.0\n"
                f"- Signaux détectés : {task.get('evidence',[])}\n"
                f"- Frames consécutives à risque : {task.get('sustained_frames',0)}\n"
                f"- Historique EMA (8 dernières) : {list(task.get('ema_history',[]))}\n\n"
                "Analyse l'image et confirme ou infirme une agression d'un humain envers un chien."
            )
        elif inc_type == 'dog_to_human':
            ctx = (
                f"CONTEXTE DÉTECTION AUTOMATIQUE :\n"
                f"- Type suspect : CHIEN agresse HUMAIN\n"
                f"- Score EMA actuel : {task.get('ema_score',0):.3f} / 1.0\n"
                f"- Score approche chien : {task.get('dog_approach_score',0):.3f} / 1.0\n"
                f"- Score fuite humain : {task.get('human_fleeing_score',0):.3f} / 1.0\n"
                f"- Facteur proximité : {task.get('proximity_factor',0):.3f} / 1.0\n"
                f"- Signaux détectés : {task.get('evidence',[])}\n"
                f"- Frames consécutives à risque : {task.get('sustained_frames',0)}\n\n"
                "Analyse l'image et confirme ou infirme une agression d'un chien envers un humain."
            )
        else:
            ctx = "Analyse cette image pour détecter toute agression entre humain et chien."

        format_req = (
            '\n\nFormat JSON attendu (pur, sans markdown) :\n'
            '{"confirmed":true|false,"confidence":0.0-1.0,'
            '"incident_type":"human_to_dog"|"dog_to_human"|"none",'
            '"severity":"minor"|"serious"|"critical",'
            '"reason":"explication courte en français",'
            '"recommended_action":"report_only"|"notify_email"|"notify_spa"'
            '|"notify_medical"|"notify_police"}'
        )

        full_prompt = system_prompt + "\n\n" + ctx + format_req

        for attempt in range(2):
            try:
                response = self.model.generate_content([pil_img, full_prompt])
                raw = response.text.strip()
                # Extraction robuste du JSON même si Gemini ajoute du texte
                m = re.search(r'\{.*\}', raw, re.DOTALL)
                if m:
                    res = json.loads(m.group())
                    res.update({'raw': raw, 'task_id': task.get('task_id'),
                                'pair_key': task.get('pair_key')})
                    return res
                logger.warning(f"LLM réponse non-JSON (tentative {attempt+1}): {raw[:80]}")
            except Exception as exc:
                logger.warning(f"LLM erreur tentative {attempt+1}/2 : {exc}")
                if attempt == 0:
                    time.sleep(2)

        # Échec total — retour neutre pour ne pas bloquer le pipeline
        return {
            'confirmed': False, 'confidence': 0.0,
            'incident_type': 'none', 'severity': 'minor',
            'reason': 'timeout_ou_erreur_api', 'recommended_action': 'report_only',
            'raw': '', 'task_id': task.get('task_id'), 'pair_key': task.get('pair_key')
        }

    def submit(self, task: dict) -> bool:
        """
        Soumet une tâche LLM. Non-bloquant.

        Args:
            task: Dict avec 'frame', 'type' et contexte de scores.
        Returns:
            True si accepté, False si queue pleine (frame ignoré silencieusement).
        """
        try:
            self.queue_in.put_nowait(task)
            return True
        except queue.Full:
            logger.debug("LLMWorker: queue_in pleine — frame sauté")
            return False

    def get_result(self) -> Optional[dict]:
        """
        Récupère un résultat LLM si disponible. Non-bloquant.

        Returns:
            Dict résultat ou None si aucun résultat disponible.
        """
        try:
            return self.queue_out.get_nowait()
        except queue.Empty:
            return None

    def stop(self):
        """Envoie le signal d'arrêt et attend la fin du thread (max 5s)."""
        self.queue_in.put(None)
        self.thread.join(timeout=5)
        logger.info("LLMWorker arrêté proprement")


# Initialisation conditionnelle : bypass propre si clé absente
if GEMINI_API_KEY:
    try:
        llm_worker  = LLMWorker(GEMINI_API_KEY)
        LLM_ENABLED = True
        print("✅ LLMWorker démarré — Gemini asynchrone (P6 corrigé)")
    except Exception as _e:
        logger.error(f"Échec LLMWorker : {_e}")
        llm_worker  = None
        LLM_ENABLED = False
        print(f"⚠️  LLMWorker désactivé : {_e}")
else:
    llm_worker  = None
    LLM_ENABLED = False
    print("⚠️  GEMINI_API_KEY absente — stage LLM bypassé (P10 corrigé)")
    print("   Ajouter la clé dans Colab → Secrets → GEMINI_API_KEY")


In [ ]:
# ─── CELLULE 09 : Agent LangGraph — Décision post-confirmation ───────────────
# Résout P7 : couche de décision structurée remplaçant trigger_agent().
# 6 nœuds + routage conditionnel sur severity : minor/serious/critical.

if not LANGGRAPH_OK:
    print("❌ LangGraph non disponible — exécuter cellule 02 puis redémarrer")
    agent_graph = None
else:
    # ── État de l'agent ───────────────────────────────────────────────────────
    class IncidentState(TypedDict):
        # Identifiants uniques
        incident_id:  str
        timestamp:    str
        video_source: str
        frame_number: int
        # Détection
        incident_type:   str     # human_to_dog | dog_to_human
        person_track_id: int
        dog_track_id:    int
        distance_px:     float
        # Scores et EMA
        risk_scores:      dict
        ema_score:        float
        ema_history:      list   # 8 dernières valeurs EMA
        sustained_frames: int    # frames consécutives >= RISK_THRESHOLD
        evidence_list:    list   # signaux détectés > seuil
        # Résultat LLM
        llm_confirmed:          bool
        llm_confidence:         float
        llm_reason:             str
        llm_severity:           str   # minor | serious | critical
        llm_recommended_action: str
        # Décisions agent
        severity:           str
        db_event_id:        int
        report_path:        str
        capture_path:       str
        annotated_frame:    bytes  # JPEG bytes du frame annoté
        notifications_sent: list
        actions_taken:      list
        routing_decision:   str   # stocker | notifier | urgences

    # ── Nœud 1 : Évaluation de la gravité ────────────────────────────────────
    def evaluer_gravite(state: IncidentState) -> dict:
        """
        Lit llm_severity et choisit la route.
        minor → stocker  |  serious → notifier  |  critical → urgences
        """
        sev  = state.get('llm_severity', 'minor')
        route = {'critical': 'urgences', 'serious': 'notifier'}.get(sev, 'stocker')
        logger.info(
            f"[AGENT] {state.get('incident_id','?')} | {sev.upper()} → {route}"
        )
        return {
            'severity':         sev,
            'routing_decision': route,
            'actions_taken':    list(state.get('actions_taken', []))
        }

    # ── Nœud 2 : Stockage ────────────────────────────────────────────────────
    def stocker_incident(state: IncidentState) -> dict:
        """
        Sauvegarde le frame annoté en JPEG et insère l'événement en base SQLite.
        Toujours exécuté quelle que soit la sévérité.
        """
        actions = list(state.get('actions_taken', []))
        cap_path = ''

        if state.get('annotated_frame'):
            fname = f"incident_{state['incident_id']}_{state['frame_number']}.jpg"
            cap_path = os.path.join(CAPTURES_DIR, fname)
            arr = cv2.imdecode(
                np.frombuffer(state['annotated_frame'], np.uint8), cv2.IMREAD_COLOR
            )
            if arr is not None:
                cv2.imwrite(cap_path, arr, [cv2.IMWRITE_JPEG_QUALITY, 95])

        db_id = save_event(DB_PATH, {**dict(state), 'capture_path': cap_path})
        actions.append('incident_stored')
        logger.info(f"[AGENT] Stocké DB id={db_id} | capture={cap_path}")
        return {'db_event_id': db_id, 'capture_path': cap_path, 'actions_taken': actions}

    # ── Nœud 3 : Rapport texte ────────────────────────────────────────────────
    def generer_rapport(state: IncidentState) -> dict:
        """
        Génère un rapport structuré en texte brut et le sauvegarde dans REPORTS_DIR.
        Toujours exécuté après stocker_incident.
        """
        actions  = list(state.get('actions_taken', []))
        evidence = state.get('evidence_list', [])

        sep = '═' * 55
        lines = [
            sep, "  RAPPORT D'INCIDENT — AGRESSION ANIMALE", sep,
            f"  ID Incident    : {state.get('incident_id','?')}",
            f"  Horodatage     : {state.get('timestamp','?')}",
            f"  Source vidéo   : {state.get('video_source','?')}",
            f"  Frame n°       : {state.get('frame_number','?')}",
            "",
            "── DÉTECTION " + "─" * 42,
            f"  Type incident  : {state.get('incident_type','?')}",
            f"  Personne #     : {state.get('person_track_id','?')}",
            f"  Chien #        : {state.get('dog_track_id','?')}",
            f"  Distance       : {state.get('distance_px',0):.1f}px",
            f"  Score EMA      : {state.get('ema_score',0):.3f} / 1.0",
            f"  Frames risque  : {state.get('sustained_frames',0)} consécutives",
            "", "  Signaux détectés :"
        ] + [f"    • {s}" for s in evidence] + [
            "", "── CONFIRMATION LLM " + "─" * 35,
            f"  Modèle         : {LLM_MODEL}",
            f"  Confirmé       : {state.get('llm_confirmed',False)}",
            f"  Confiance      : {state.get('llm_confidence',0):.0%}",
            f"  Sévérité       : {state.get('llm_severity','?').upper()}",
            f"  Motif          : {state.get('llm_reason','?')}",
            "", "── DÉCISION AGENT " + "─" * 37,
            "  Actions prises :"
        ] + [f"    ✓ {a}" for a in actions] + [
            "", "── FICHIERS " + "─" * 43,
            f"  Capture        : {state.get('capture_path','?')}",
            sep
        ]

        os.makedirs(REPORTS_DIR, exist_ok=True)
        report_path = os.path.join(REPORTS_DIR, f"report_{state.get('incident_id','x')}.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write("\n".join(lines))

        actions.append('report_generated')
        logger.info(f"[AGENT] Rapport : {report_path}")
        return {'report_path': report_path, 'actions_taken': actions}

    # ── Nœud 4 : Email ────────────────────────────────────────────────────────
    def notifier_email(state: IncidentState) -> dict:
        """
        Envoie un email d'alerte avec rapport et capture en PJ.
        Si EMAIL_ENABLED=False, loggue 'email_skipped' sans erreur.
        """
        actions = list(state.get('actions_taken', []))
        notifs  = list(state.get('notifications_sent', []))

        if not EMAIL_ENABLED:
            actions.append('email_skipped')
            return {'actions_taken': actions, 'notifications_sent': notifs}

        try:
            msg = MIMEMultipart()
            msg['From']    = SMTP_USER
            msg['To']      = ', '.join(ALERT_RECIPIENTS)
            msg['Subject'] = (
                f"ALERTE [{state.get('severity','?').upper()}] "
                f"{state.get('incident_type','?')} — {state.get('timestamp','?')}"
            )
            rp = state.get('report_path', '')
            body = open(rp, encoding='utf-8').read() if rp and os.path.exists(rp) else (
                f"Incident {state.get('incident_id')} détecté."
            )
            msg.attach(MIMEText(body, 'plain', 'utf-8'))

            # PJ1 : capture JPEG
            cp = state.get('capture_path', '')
            if cp and os.path.exists(cp):
                with open(cp, 'rb') as f:
                    msg.attach(MIMEImage(f.read(), name=Path(cp).name))

            # PJ2 : rapport texte en pièce jointe
            if rp and os.path.exists(rp):
                from email.mime.base import MIMEBase
                from email import encoders
                with open(rp, 'rb') as f:
                    part = MIMEBase('application', 'octet-stream')
                    part.set_payload(f.read())
                encoders.encode_base64(part)
                part.add_header('Content-Disposition', f'attachment; filename={Path(rp).name}')
                msg.attach(part)

            with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=10) as srv:
                srv.starttls()
                srv.login(SMTP_USER, SMTP_PASS)
                srv.sendmail(SMTP_USER, ALERT_RECIPIENTS, msg.as_string())

            actions.append('email_sent'); notifs.append('email')
            logger.info("[AGENT] Email envoyé")
        except Exception as exc:
            actions.append('email_failed')
            logger.error(f"[AGENT] Échec email : {exc}")

        return {'actions_taken': actions, 'notifications_sent': notifs}

    # ── Nœud 5 : Urgences ─────────────────────────────────────────────────────
    def contacter_urgences(state: IncidentState) -> dict:
        """
        Logique différenciée :
        H→D : SPA + (critical → log police)
        D→H : serious → médical | critical → médical + log police
        """
        actions  = list(state.get('actions_taken', []))
        notifs   = list(state.get('notifications_sent', []))
        inc_type = state.get('incident_type', '')
        sev      = state.get('severity', '')

        def _urgent_email(dest, subject_suffix):
            if not EMAIL_ENABLED or not dest:
                return False
            try:
                m = MIMEMultipart()
                m['From']    = SMTP_USER
                m['To']      = dest
                m['Subject'] = f"URGENT [{sev.upper()}] {subject_suffix}"
                rp = state.get('report_path', '')
                body = open(rp, encoding='utf-8').read() if rp and os.path.exists(rp) else ''
                m.attach(MIMEText(body, 'plain', 'utf-8'))
                with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=10) as srv:
                    srv.starttls(); srv.login(SMTP_USER, SMTP_PASS)
                    srv.sendmail(SMTP_USER, [dest], m.as_string())
                return True
            except Exception as e:
                logger.error(f"[AGENT] Email urgent échoué : {e}")
                return False

        if inc_type == 'human_to_dog':
            ok = _urgent_email(SPA_EMAIL, f"Maltraitance animale — {state.get('timestamp')}")
            actions.append('spa_notified' if ok else 'spa_failed')
            notifs.append('spa')
            if sev == 'critical':
                logger.warning(f"[AGENT] POLICE ALERT — {state.get('incident_id')}")
                actions.append('police_alert_logged')

        elif inc_type == 'dog_to_human':
            if sev in ('serious', 'critical'):
                ok = _urgent_email(MEDICAL_EMAIL, f"Morsure potentielle — {state.get('timestamp')}")
                actions.append('medical_notified' if ok else 'medical_failed')
                notifs.append('medical')
            if sev == 'critical':
                logger.warning(f"[AGENT] POLICE ALERT — {state.get('incident_id')}")
                actions.append('police_alert_logged')

        logger.info(f"[AGENT] Urgences : {actions}")
        return {'actions_taken': actions, 'notifications_sent': notifs}

    # ── Nœud 6 : Faux positif ─────────────────────────────────────────────────
    def log_non_confirme(state: IncidentState) -> dict:
        """Enregistre les détections non confirmées par le LLM pour analyse."""
        actions = list(state.get('actions_taken', []))
        save_event(DB_PATH, {**dict(state), 'llm_confirmed': False, 'severity': 'none'})
        actions.append('false_positive_logged')
        logger.info(f"[AGENT] Faux positif — paire ({state.get('person_track_id')}, {state.get('dog_track_id')})")
        return {'actions_taken': actions}

    # ── Compilation du graphe ──────────────────────────────────────────────────
    builder = StateGraph(IncidentState)
    for name, fn in [
        ("evaluer_gravite",    evaluer_gravite),
        ("stocker_incident",   stocker_incident),
        ("generer_rapport",    generer_rapport),
        ("notifier_email",     notifier_email),
        ("contacter_urgences", contacter_urgences),
        ("log_non_confirme",   log_non_confirme),
    ]:
        builder.add_node(name, fn)

    builder.set_entry_point("evaluer_gravite")
    # Flux séquentiel obligatoire
    builder.add_edge("evaluer_gravite",  "stocker_incident")
    builder.add_edge("stocker_incident", "generer_rapport")
    # Routage conditionnel après rapport
    builder.add_conditional_edges(
        "generer_rapport",
        lambda s: s["routing_decision"],
        {"stocker": END, "notifier": "notifier_email", "urgences": "notifier_email"}
    )
    # Routage conditionnel après email
    builder.add_conditional_edges(
        "notifier_email",
        lambda s: s["routing_decision"],
        {"notifier": END, "urgences": "contacter_urgences"}
    )
    builder.add_edge("contacter_urgences", END)
    builder.add_edge("log_non_confirme",   END)

    agent_graph = builder.compile()
    print("✅ Agent LangGraph compilé (P7 corrigé)")
    print("   Flux : evaluer_gravite → stocker → rapport → [email] → [urgences]")
    print("   Routes : minor=stocker | serious=email | critical=email+urgences")

In [ ]:
# ─── CELLULE 10 : Module notifications — Email + Rapport ─────────────────────
# Fonctions standalone utilisées aussi par l'agent LangGraph (cellule 09).

def send_email_alert(subject: str, body: str,
                     attachments: list = None) -> bool:
    """
    Envoie un email via SMTP avec des pièces jointes optionnelles.

    Args:
        subject: Objet de l'email.
        body: Corps en texte brut.
        attachments: Liste de chemins de fichiers à joindre.
    Returns:
        True si envoi réussi, False sinon.
    """
    if not EMAIL_ENABLED:
        logger.info("Email désactivé (EMAIL_ENABLED=False)")
        return False
    if not SMTP_USER or not SMTP_PASS:
        logger.warning("SMTP_USER ou SMTP_PASS manquant")
        return False
    try:
        msg            = MIMEMultipart()
        msg['From']    = SMTP_USER
        msg['To']      = ', '.join(ALERT_RECIPIENTS)
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain', 'utf-8'))

        for path in (attachments or []):
            if not os.path.exists(path):
                continue
            ext = Path(path).suffix.lower()
            with open(path, 'rb') as fh:
                if ext in ('.jpg', '.jpeg', '.png'):
                    msg.attach(MIMEImage(fh.read(), name=Path(path).name))
                else:
                    from email.mime.base import MIMEBase
                    from email import encoders as enc
                    part = MIMEBase('application', 'octet-stream')
                    part.set_payload(fh.read())
                    enc.encode_base64(part)
                    part.add_header('Content-Disposition',
                                    f'attachment; filename={Path(path).name}')
                    msg.attach(part)

        with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=10) as srv:
            srv.starttls()
            srv.login(SMTP_USER, SMTP_PASS)
            srv.sendmail(SMTP_USER, ALERT_RECIPIENTS, msg.as_string())

        logger.info(f"Email envoyé : {subject}")
        return True
    except Exception as exc:
        logger.error(f"Échec email : {exc}")
        return False


def generate_text_report(state: dict) -> str:
    """
    Génère un rapport d'incident formaté en texte brut.

    Args:
        state: Dict compatible IncidentState.
    Returns:
        Chemin du fichier rapport créé.
    """
    evidence = state.get('evidence_list', [])
    actions  = state.get('actions_taken', [])
    sep = '═' * 55
    lines = [
        sep, "  RAPPORT D'INCIDENT — AGRESSION ANIMALE", sep,
        f"  ID Incident    : {state.get('incident_id','?')}",
        f"  Horodatage     : {state.get('timestamp','?')}",
        f"  Source vidéo   : {state.get('video_source','?')}",
        f"  Frame n°       : {state.get('frame_number','?')}",
        "", "── DÉTECTION " + "─" * 42,
        f"  Type           : {state.get('incident_type','?')}",
        f"  Personne #     : {state.get('person_track_id','?')}",
        f"  Chien #        : {state.get('dog_track_id','?')}",
        f"  Distance bords : {state.get('distance_px',0):.1f}px",
        f"  Score EMA      : {state.get('ema_score',0):.3f} / 1.0",
        f"  Frames risque  : {state.get('sustained_frames',0)} consécutives",
        "", "  Signaux détectés :"
    ] + [f"    • {s}" for s in evidence] + [
        "", "── CONFIRMATION LLM " + "─" * 35,
        f"  Modèle         : {LLM_MODEL}",
        f"  Confirmé       : {state.get('llm_confirmed',False)}",
        f"  Confiance      : {state.get('llm_confidence',0):.0%}",
        f"  Sévérité       : {state.get('llm_severity','?').upper()}",
        f"  Motif          : {state.get('llm_reason','?')}",
        "", "── ACTIONS AGENT " + "─" * 38,
    ] + [f"    ✓ {a}" for a in actions] + [
        "", "── FICHIERS " + "─" * 43,
        f"  Capture        : {state.get('capture_path','?')}",
        sep
    ]
    os.makedirs(REPORTS_DIR, exist_ok=True)
    path = os.path.join(REPORTS_DIR, f"report_{state.get('incident_id','x')}.txt")
    with open(path, 'w', encoding='utf-8') as f:
        f.write("\n".join(lines))
    return path


print("✅ Module notifications prêt (email + rapport texte)")

In [ ]:
# ─── CELLULE 11 : Module visualisation — Annotation des frames ───────────────
# Dessine les bboxes, scores EMA, keypoints et alertes sur chaque frame.

# Palette couleurs (BGR) indexée par niveau de risque
_COLOR_LOW    = (0, 200, 0)    # Vert — pas de risque
_COLOR_MEDIUM = (0, 165, 255)  # Orange — surveillance
_COLOR_HIGH   = (0, 0, 255)    # Rouge — alerte
_COLOR_PERSON = (255, 180, 0)  # Bleu-cyan — personne trackée
_COLOR_DOG    = (0, 255, 140)  # Vert-clair — chien tracké


def _risk_color(score: float) -> tuple:
    """Retourne la couleur BGR correspondant au niveau de risque."""
    if score >= RISK_THRESHOLD:
        return _COLOR_HIGH
    if score >= RISK_THRESHOLD * 0.6:
        return _COLOR_MEDIUM
    return _COLOR_LOW


def annotate_frame(
    frame:    np.ndarray,
    persons:  list,
    dogs:     list,
    ema_scores: dict,
    sustained_counters: dict,
    risk_data_map: dict = None
) -> np.ndarray:
    """
    Annote un frame avec les détections, scores EMA et alertes visuelles.

    Args:
        frame: Frame BGR original (non modifié).
        persons: Liste de dicts {track_id, bbox, keypoints}.
        dogs: Liste de dicts {track_id, bbox}.
        ema_scores: {(pid,did,'h2d'|'d2h'): float} — scores EMA courants.
        sustained_counters: {(pid,did): int} — frames consécutives >= seuil.
        risk_data_map: {(pid,did): dict} — sortie complète de compute_pair().
    Returns:
        Frame annoté (copie).
    """
    out = frame.copy()
    h, w = out.shape[:2]

    # ── Chiens ────────────────────────────────────────────────────────────────
    for dog in dogs:
        x1, y1, x2, y2 = [int(v) for v in dog['bbox']]
        cv2.rectangle(out, (x1,y1), (x2,y2), _COLOR_DOG, 2)
        cv2.putText(out, f"Dog#{dog['track_id']}", (x1, y1-6),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, _COLOR_DOG, 1)

    # ── Personnes + scores ────────────────────────────────────────────────────
    for p in persons:
        pid = p['track_id']
        x1, y1, x2, y2 = [int(v) for v in p['bbox']]

        # Score EMA dominant pour cette personne (max sur tous les chiens)
        max_ema = 0.0
        for dog in dogs:
            did = dog['track_id']
            max_ema = max(
                max_ema,
                ema_scores.get(('h2d', pid, did), 0.0),
                ema_scores.get(('d2h', pid, did), 0.0)
            )

        color = _risk_color(max_ema)
        cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
        label = f"P#{pid} EMA:{max_ema:.2f}"
        cv2.putText(out, label, (x1, y1-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # Keypoints (poignets et épaules mis en avant)
        kps = p.get('keypoints')
        if kps is not None:
            for idx in [KP_SHOULDER_L, KP_SHOULDER_R, KP_ELBOW_L, KP_ELBOW_R,
                        KP_WRIST_L, KP_WRIST_R]:
                if idx < len(kps) and kps[idx][2] > 0.3:
                    kx, ky = int(kps[idx][0]), int(kps[idx][1])
                    r = 5 if idx in (KP_WRIST_L, KP_WRIST_R) else 3
                    cv2.circle(out, (kx,ky), r, color, -1)

        # Alerte visuelle si sustained >= MIN_SUSTAINED_FRAMES
        for dog in dogs:
            did = dog['track_id']
            sust = sustained_counters.get((pid, did), 0)
            if sust >= MIN_SUSTAINED_FRAMES:
                dx1,dy1,dx2,dy2 = [int(v) for v in dog['bbox']]
                pc = ((x1+x2)//2, (y1+y2)//2)
                dc = ((dx1+dx2)//2, (dy1+dy2)//2)
                cv2.line(out, pc, dc, _COLOR_HIGH, 2)
                alert_txt = f"RISQUE x{sust}"
                cv2.putText(out, alert_txt, ((pc[0]+dc[0])//2, (pc[1]+dc[1])//2 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, _COLOR_HIGH, 2)

    # ── HUD — stats en bas ────────────────────────────────────────────────────
    hud = f"Personnes:{len(persons)}  Chiens:{len(dogs)}  Seuil EMA:{RISK_THRESHOLD}"
    cv2.putText(out, hud, (8, h-12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200,200,200), 1)

    return out


print("✅ Module visualisation prêt")

In [ ]:
# ─── CELLULE 12 : Pipeline principal two-stage intégré ───────────────────────
# Stage 0 : sentinelle YOLOv8n (~2ms)
# Stage 1 : détection YOLOv8l + ByteTrack + pose
# Stage 2 : RiskScorerV2 + EMA
# Stage 3 : LLMWorker asynchrone
# Stage 4 : Agent LangGraph

def run_pipeline(
    video_source,
    max_frames:    int  = None,
    save_output:   bool = True,
    output_path:   str  = '/content/output_annotated.mp4',
    display_every: int  = 30
) -> dict:
    """
    Pipeline complet de détection d'agression animale two-stage.

    Args:
        video_source: Chemin fichier vidéo, URL stream ou entier (webcam).
        max_frames: Limite de frames à traiter (None = jusqu'à la fin).
        save_output: Écrit la vidéo annotée si True.
        output_path: Chemin de la vidéo de sortie.
        display_every: Affiche les stats toutes les N frames.
    Returns:
        Dict de statistiques de traitement.
    """
    # ── Ouverture source ───────────────────────────────────────────────────────
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d'ouvrir : {video_source}")

    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"📹 Source : {video_source}  |  {width}×{height} @ {fps:.1f}fps  |  {total} frames")

    # ── Writer vidéo output ────────────────────────────────────────────────────
    writer = None
    if save_output:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        print(f"   Output : {output_path}")

    # ── Thread de lecture vidéo (évite les blocages I/O) ──────────────────────
    frame_queue = queue.Queue(maxsize=8)

    def _reader():
        while True:
            ret, fr = cap.read()
            if not ret:
                frame_queue.put(None)  # Signal de fin
                break
            frame_queue.put(fr)

    reader_thread = threading.Thread(target=_reader, daemon=True)
    reader_thread.start()

    # ── Variables d'état ───────────────────────────────────────────────────────
    ema_scores        = {}  # {('h2d'|'d2h', pid, did): float}
    ema_history       = defaultdict(lambda: deque(maxlen=8))
    sustained         = defaultdict(int)   # {(pid,did): frames consécutives >= seuil}
    last_llm_frame    = defaultdict(lambda: -MIN_FRAMES_BETWEEN_LLM)
    llm_pending       = {}                 # {(pid,did): bool}
    risk_data_map     = {}

    stats = {
        'frames_total':    0,    'frames_sentinel_pass': 0,
        'llm_calls':       0,    'alerts_confirmed':     0,
        'false_positives': 0,    'start_time':           time.time()
    }

    frame_idx = 0
    print("🚀 Pipeline démarré...")

    try:
        while True:
            frame = frame_queue.get(timeout=30)
            if frame is None:
                break  # Fin du flux
            if max_frames and frame_idx >= max_frames:
                break

            frame_idx          += 1
            stats['frames_total'] += 1

            # ── STAGE 0 : Sentinelle ──────────────────────────────────────────
            with torch.no_grad():
                sentinel_res = model_sentinel(
                    frame,
                    classes=[PERSON_CLASS_ID, DOG_CLASS_ID],
                    conf=SENTINEL_CONF,
                    verbose=False
                )

            classes_det = set()
            if sentinel_res[0].boxes is not None:
                classes_det = set(sentinel_res[0].boxes.cls.cpu().tolist())

            # Si la sentinelle ne voit pas person ET dog → frame brut écrit
            if not (PERSON_CLASS_ID in classes_det and DOG_CLASS_ID in classes_det):
                if writer:
                    writer.write(frame)
                continue

            stats['frames_sentinel_pass'] += 1

            # ── STAGE 1 : Détection lourde + ByteTrack + Pose ────────────────
            with torch.no_grad():
                detect_res = model_detect.track(
                    frame, persist=True,
                    classes=[PERSON_CLASS_ID, DOG_CLASS_ID],
                    conf=DETECT_CONF, verbose=False,
                    tracker='bytetrack.yaml'
                )
                pose_res = model_pose(frame, conf=POSE_CONF, verbose=False)

            # Purge GPU périodique pour éviter la fragmentation mémoire
            if frame_idx % 100 == 0 and DEVICE == 'cuda':
                torch.cuda.empty_cache()

            # Parse des détections
            persons, dogs = [], []
            if detect_res[0].boxes is not None:
                boxes = detect_res[0].boxes
                for i in range(len(boxes)):
                    cls_id   = int(boxes.cls[i].item())
                    track_id = int(boxes.id[i].item()) if boxes.id is not None else i
                    bbox     = boxes.xyxy[i].cpu().numpy()
                    if cls_id == PERSON_CLASS_ID:
                        persons.append({'track_id': track_id, 'bbox': bbox, 'keypoints': None})
                    elif cls_id == DOG_CLASS_ID:
                        dogs.append({'track_id': track_id, 'bbox': bbox})

            if not persons or not dogs:
                if writer:
                    writer.write(annotate_frame(frame, persons, dogs, ema_scores, sustained))
                continue

            # Correspondance pose → personnes par IoU
            kp_map = {}
            if pose_res[0].keypoints is not None and pose_res[0].boxes is not None:
                kps_data  = pose_res[0].keypoints.data.cpu().numpy()
                pose_boxes = pose_res[0].boxes.xyxy.cpu().numpy()
                for p in persons:
                    pb = p['bbox']; best_iou = 0.2; best_kp = None
                    for j, pb2 in enumerate(pose_boxes):
                        ix1 = max(pb[0],pb2[0]); iy1 = max(pb[1],pb2[1])
                        ix2 = min(pb[2],pb2[2]); iy2 = min(pb[3],pb2[3])
                        inter = max(0,ix2-ix1)*max(0,iy2-iy1)
                        a1 = (pb[2]-pb[0])*(pb[3]-pb[1])
                        a2 = (pb2[2]-pb2[0])*(pb2[3]-pb2[1])
                        iou = inter/(a1+a2-inter) if (a1+a2-inter) > 0 else 0
                        if iou > best_iou:
                            best_iou = iou
                            best_kp  = kps_data[j] if j < len(kps_data) else None
                    if best_kp is not None:
                        kp_map[p['track_id']] = best_kp
                        p['keypoints'] = best_kp

            # ── STAGE 2 : RiskScorerV2 + EMA ─────────────────────────────────
            for p in persons:
                pid  = p['track_id']
                kps  = kp_map.get(pid)
                for dog in dogs:
                    did = dog['track_id']
                    tk  = (pid, did)

                    rd = risk_scorer.compute_pair(pid, kps, p['bbox'], did, dog['bbox'])
                    risk_data_map[tk] = rd

                    # Mise à jour du dictionnaire EMA local (pour visualisation)
                    ema_scores[('h2d', pid, did)] = rd['human_to_dog']['ema_score']
                    ema_scores[('d2h', pid, did)] = rd['dog_to_human']['ema_score']
                    ema_history[tk].append(rd['dominant_ema'])

                    dom_ema = rd['dominant_ema']

                    # EMA >= seuil → incrémenter sustained, sinon décrémenter doucement
                    if dom_ema >= RISK_THRESHOLD:
                        sustained[tk] += 1
                    else:
                        sustained[tk] = max(0, sustained[tk] - 1)

                    # ── STAGE 3 : Soumettre au LLM si conditions remplies ─────
                    frames_since = frame_idx - last_llm_frame[tk]
                    already_pending = llm_pending.get(tk, False)

                    if (LLM_ENABLED
                            and sustained[tk] >= MIN_SUSTAINED_FRAMES
                            and frames_since > MIN_FRAMES_BETWEEN_LLM
                            and not already_pending):

                        inc_type = rd['dominant_type']
                        details  = rd[inc_type]
                        task = {
                            'task_id':          str(uuid.uuid4()),
                            'pair_key':         tk,
                            'frame':            frame.copy(),
                            'type':             inc_type,
                            'ema_score':        dom_ema,
                            'ema_history':      list(ema_history[tk]),
                            'sustained_frames': sustained[tk],
                            'proximity_factor': rd['proximity_factor'],
                            'evidence':         details['evidence'],
                            # Champs spécifiques H→D
                            'gesture_score':    details.get('raw_gesture_score', 0),
                            # Champs spécifiques D→H
                            'dog_approach_score':  details.get('details', {}).get('dog_approach_score', 0),
                            'human_fleeing_score': details.get('details', {}).get('human_fleeing_score', 0),
                        }
                        if llm_worker.submit(task):
                            llm_pending[tk]    = True
                            last_llm_frame[tk] = frame_idx
                            stats['llm_calls'] += 1
                            print(f"\n🔍 LLM soumis — P#{pid} D#{did} "
                                  f"EMA={dom_ema:.3f} type={inc_type}")

            # ── STAGE 4 : Récupérer résultats LLM (non bloquant) ─────────────
            while LLM_ENABLED:
                result = llm_worker.get_result()
                if result is None:
                    break

                res_key = result.get('pair_key')
                llm_pending[res_key] = False

                if result.get('confirmed', False):
                    stats['alerts_confirmed'] += 1
                    rd  = risk_data_map.get(res_key, {})
                    inc_type = rd.get('dominant_type', 'unknown')
                    pid, did = res_key

                    # Construction de l'IncidentState pour le graphe LangGraph
                    inc_id  = result.get('task_id', str(uuid.uuid4()))
                    dom_det = rd.get(inc_type, {})

                    # Encodage du frame annoté en JPEG bytes pour l'agent
                    annotated = annotate_frame(frame, persons, dogs, ema_scores, sustained)
                    _, jpg_buf = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 92])

                    state = {
                        'incident_id':      inc_id,
                        'timestamp':        datetime.now().isoformat(),
                        'video_source':     str(video_source),
                        'frame_number':     frame_idx,
                        'incident_type':    inc_type,
                        'person_track_id':  pid,
                        'dog_track_id':     did,
                        'distance_px':      rd.get('distance_px', 0),
                        'risk_scores':      rd,
                        'ema_score':        rd.get('dominant_ema', 0),
                        'ema_history':      list(ema_history[res_key]),
                        'sustained_frames': sustained.get(res_key, 0),
                        'evidence_list':    dom_det.get('evidence', []),
                        'llm_confirmed':    True,
                        'llm_confidence':   result.get('confidence', 0),
                        'llm_reason':       result.get('reason', ''),
                        'llm_severity':     result.get('severity', 'minor'),
                        'llm_recommended_action': result.get('recommended_action', 'report_only'),
                        'llm_model':        LLM_MODEL,
                        'severity':         '',   # Sera rempli par evaluer_gravite
                        'db_event_id':      -1,
                        'report_path':      '',
                        'capture_path':     '',
                        'annotated_frame':  jpg_buf.tobytes(),
                        'notifications_sent': [],
                        'actions_taken':    [],
                        'routing_decision': '',
                        'processing_time_ms': 0.0
                    }

                    # Invocation synchrone de l'agent (dans le thread principal)
                    if LANGGRAPH_OK and agent_graph:
                        final = agent_graph.invoke(state)
                        print(f"\n{'═'*55}")
                        print(f"🚨 INCIDENT CONFIRMÉ — {inc_type.upper()}")
                        print(f"   Sévérité  : {final.get('severity','?').upper()}")
                        print(f"   Actions   : {final.get('actions_taken',[])} ")
                        print(f"   Rapport   : {final.get('report_path','?')}")
                        print(f"{'═'*55}\n")
                    else:
                        # Fallback si LangGraph absent
                        save_event(DB_PATH, state)
                        print(f"🚨 INCIDENT confirmé (sans agent) — stocké en DB")

                    # Réinitialisation de la paire après confirmation
                    sustained[res_key]   = 0
                    for k in [('h2d',)+res_key, ('d2h',)+res_key]:
                        risk_scorer.ema_scores[k] = 0.0

                else:
                    # LLM infirme → décrémentation douce du compteur
                    stats['false_positives'] += 1
                    sustained[res_key] = max(0, sustained.get(res_key, 0) - 4)
                    print(f"   LLM: non confirmé — {result.get('reason','?')[:60]}")

            # ── Annotation et écriture du frame ───────────────────────────────
            annotated = annotate_frame(frame, persons, dogs, ema_scores, sustained)
            if writer:
                writer.write(annotated)

            # Affichage périodique des stats
            if frame_idx % display_every == 0:
                elapsed = time.time() - stats['start_time']
                fps_proc = frame_idx / max(elapsed, 0.001)
                sent_pct = 100 * stats['frames_sentinel_pass'] / max(stats['frames_total'],1)
                print(f"Frame {frame_idx:5d} | {fps_proc:.1f}fps | "
                      f"Sentinelle:{sent_pct:.0f}% pass | "
                      f"LLM:{stats['llm_calls']} | Alertes:{stats['alerts_confirmed']}")

    except KeyboardInterrupt:
        print("\n⛔ Pipeline interrompu par l'utilisateur")
    finally:
        cap.release()
        if writer:
            writer.release()
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    elapsed = time.time() - stats['start_time']
    stats['elapsed_sec'] = round(elapsed, 1)
    stats['avg_fps']     = round(frame_idx / max(elapsed, 0.001), 1)
    print(f"\n✅ Pipeline terminé — {frame_idx} frames en {elapsed:.1f}s ({stats['avg_fps']}fps)")
    print(f"   Sentinelle pass : {stats['frames_sentinel_pass']}/{stats['frames_total']}")
    print(f"   LLM calls      : {stats['llm_calls']}")
    print(f"   Alertes        : {stats['alerts_confirmed']}")
    return stats


print("✅ Pipeline principal prêt (stages 0→4)")
print("   P1: sentinelle YOLOv8n | P6: Gemini async (gratuit) | P7: agent LangGraph")

In [ ]:
# ─── CELLULE 13 : Sélection de la source vidéo ───────────────────────────────
# Choisir un seul bloc SOURCE_TYPE parmi les options ci-dessous.

import os
import cv2
import numpy as np

# ─── OPTION : SOURCE_TYPE ────────────────────────────────────────────────────
# "youtube"   → téléchargement yt-dlp
# "drive"     → fichier depuis Google Drive
# "upload"    → upload depuis la machine locale
# "synthetic" → vidéo générée synthétiquement (test sans source externe)
SOURCE_TYPE = "synthetic"

VIDEO_SOURCE = None

# ── Option 1 : YouTube ────────────────────────────────────────────────────────
if SOURCE_TYPE == "youtube":
    !pip install -q yt-dlp
    YOUTUBE_URL = "https://www.youtube.com/watch?v=REMPLACER_PAR_URL"
    !yt-dlp -f "best[ext=mp4][height<=720]" -o "/content/video_input.mp4" "{YOUTUBE_URL}"
    VIDEO_SOURCE = "/content/video_input.mp4"

# ── Option 2 : Google Drive ───────────────────────────────────────────────────
elif SOURCE_TYPE == "drive":
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PATH = "/content/drive/MyDrive/REMPLACER_PAR_CHEMIN.mp4"
    VIDEO_SOURCE = DRIVE_PATH

# ── Option 3 : Upload local ───────────────────────────────────────────────────
elif SOURCE_TYPE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_SOURCE = list(uploaded.keys())[0]
        print(f"Fichier uploadé : {VIDEO_SOURCE}")

# ── Option 4 : Vidéo synthétique (test sans source réelle) ───────────────────
elif SOURCE_TYPE == "synthetic":
    synth_path = "/content/synthetic_test.mp4"
    fps_s, w_s, h_s = 25, 640, 480
    out_s = cv2.VideoWriter(synth_path, cv2.VideoWriter_fourcc(*'mp4v'), fps_s, (w_s, h_s))
    for i in range(250):
        f = np.zeros((h_s, w_s, 3), dtype=np.uint8) + 50
        # Personne (rectangle blanc en mouvement)
        px = 100 + i; py = 200
        cv2.rectangle(f, (px, py), (px+60, py+120), (200,200,200), -1)
        # Chien (rectangle brun rapproché)
        dx = 200 + i//2; dy = 240
        cv2.rectangle(f, (dx, dy), (dx+50, dy+40), (40,80,140), -1)
        cv2.putText(f, f"Frame {i}", (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)
        out_s.write(f)
    out_s.release()
    VIDEO_SOURCE = synth_path
    print(f"✅ Vidéo synthétique créée : {synth_path} (250 frames)")

# ── Validation ────────────────────────────────────────────────────────────────
if VIDEO_SOURCE and os.path.exists(str(VIDEO_SOURCE)):
    cap_test = cv2.VideoCapture(VIDEO_SOURCE)
    ok = cap_test.isOpened()
    cap_test.release()
    print(f"{'✅' if ok else '❌'} Source vidéo : {VIDEO_SOURCE}")
else:
    print(f"⚠️  SOURCE_TYPE='{SOURCE_TYPE}' — VIDEO_SOURCE non défini ou fichier introuvable")

In [ ]:
import math,json,uuid,time,os
from pathlib import Path
from datetime import datetime
from typing import List
import numpy as np
import cv2
from PIL import Image

OUT_DIR=Path(OUTPUT_DIR)
KEYFRAME_DIR=OUT_DIR/'keyframes'
REPORT_DIR=OUT_DIR/'reports'
for d in(KEYFRAME_DIR,REPORT_DIR):d.mkdir(parents=True,exist_ok=True)

def make_mosaic(frames,max_frames=9,cell_size=256):
    frames=frames[:max_frames];n=len(frames)
    cols=math.ceil(math.sqrt(n));rows=math.ceil(n/cols)
    canvas=np.zeros((rows*cell_size,cols*cell_size,3),dtype=np.uint8)
    for idx,frm in enumerate(frames):
        r,c=divmod(idx,cols);thumb=cv2.resize(frm,(cell_size,cell_size))
        canvas[r*cell_size:(r+1)*cell_size,c*cell_size:(c+1)*cell_size]=thumb
    return canvas

def save_keyframes(frames,incident_id):
    path=KEYFRAME_DIR/f'{incident_id}_mosaic.jpg'
    cv2.imwrite(str(path),make_mosaic(frames),[cv2.IMWRITE_JPEG_QUALITY,88])
    return str(path)

def save_report(llm_result,incident_id,person_id,dog_id,ema_risk,timestamp,
                sustained_frames=0,video_source=''):
    # sustained_frames + video_source added for forward_incident() compatibility
    report={'incident_id':incident_id,
            'timestamp_iso':datetime.utcfromtimestamp(timestamp).isoformat()+'Z',
            'video_source':video_source or str(globals().get('VIDEO_SOURCE','')),
            'person_track_id':person_id,'dog_track_id':dog_id,
            'ema_risk_at_trigger':round(ema_risk,4),
            'sustained_frames':sustained_frames,
            'llm_model':LLM_MODEL,'llm_result':llm_result}
    path=REPORT_DIR/f'{incident_id}_report.json'
    path.write_text(json.dumps(report,indent=2))
    return str(path)

def display_keyframes_notebook(mosaic,llm_result,person_id,dog_id):
    from IPython.display import display,Markdown,Image as I
    import io
    c=llm_result.get('is_aggression',False);conf=llm_result.get('confidence',0.)
    s=llm_result.get('severity','none');a=llm_result.get('aggression_type','none')
    obs=llm_result.get('key_observations',[])
    rm=llm_result.get('report_markdown','')
    hdr=(f"### {'CONFIRMED' if c else 'NOT confirmed'} P#{person_id}/D#{dog_id}\n"
         f"|Field|Value|\n|---|---|\n|Conf|{conf:.0%}|\n|Sev|{s}|\n|Type|{a}|\n")
    if obs:hdr+='\n'.join(f'- {o}' for o in obs)
    if rm:hdr+=f'\n{rm}'
    display(Markdown(hdr))
    buf=io.BytesIO();Image.fromarray(mosaic[...,::-1]).save(buf,format='PNG')
    buf.seek(0);display(I(data=buf.read()))

print('Output helpers ready, saving to:',str(OUT_DIR))


In [ ]:
# ─── CELLULE 15 : Dashboard — Stats et consultation des incidents ─────────────
# Affiche un résumé des incidents en base et les derniers événements.

stats = get_stats_summary()
print("═" * 55)
print("  DASHBOARD — BASE DE DONNÉES INCIDENTS")
print("═" * 55)
print(f"  Total incidents     : {stats['total']}")
print(f"  Confirmés LLM       : {stats['confirmed']}")
print(f"  Faux positifs       : {stats['false_positives']}")
print()
print("  Par type :")
for t, n in stats.get('by_type', {}).items():
    print(f"    {t:<20} : {n}")
print()
print("  Par sévérité :")
for s, n in stats.get('by_severity', {}).items():
    print(f"    {s:<20} : {n}")
print("═" * 55)

# Derniers incidents confirmés
print("\n── 10 derniers incidents confirmés ───────────────────")
recent = get_incidents_by_severity('serious', 5) + get_incidents_by_severity('critical', 5)
recent.sort(key=lambda x: x.get('id', 0), reverse=True)
if recent:
    for ev in recent[:10]:
        print(
            f"  [{ev.get('id','?'):4d}] {ev.get('timestamp','?')[:19]}  "
            f"{ev.get('incident_type','?'):<14}  "
            f"sev={ev.get('severity','?'):<8}  "
            f"EMA={ev.get('ema_score',0):.3f}  "
            f"conf={ev.get('llm_confidence',0):.0%}"
        )
else:
    print("  (aucun incident confirmé pour l'instant)")

print("\n── Consultation par type ──────────────────────────────")
print("  Exemples d'appels disponibles :")
print("    get_incidents_by_type('human_to_dog', limit=5)")
print("    get_incidents_by_type('dog_to_human', limit=5)")
print("    get_incident_by_id('<uuid>')")
print("    export_to_csv('/content/exports.csv')")

In [ ]:
# ─── CELLULE 16 : Tests unitaires ────────────────────────────────────────────
# 8 tests couvrant RiskScorerV2 et le routage de l'agent LangGraph.
# Chaque test affiche ✅ si réussi, ❌ + détail si échoué.

import traceback

_scorer_test = RiskScorerV2(history_len=8)
_pass = 0; _fail = 0

def _test(name, fn):
    global _pass, _fail
    try:
        fn()
        print(f"  ✅ {name}")
        _pass += 1
    except AssertionError as e:
        print(f"  ❌ {name} — {e}")
        _fail += 1
    except Exception as e:
        print(f"  ❌ {name} — Exception : {e}")
        traceback.print_exc()
        _fail += 1

def _make_kps(shoulder_y=250, wrist_y=200, wrist_x=320, conf=0.9):
    """Crée un tableau de keypoints synthétiques [17, 3]."""
    kps = np.zeros((17, 3))
    kps[:, 2] = 0.1   # Conf basse par défaut
    kps[KP_SHOULDER_L] = [300, shoulder_y, conf]
    kps[KP_SHOULDER_R] = [340, shoulder_y, conf]
    kps[KP_ELBOW_L]    = [300, (shoulder_y+wrist_y)//2, conf]
    kps[KP_ELBOW_R]    = [340, (shoulder_y+wrist_y)//2, conf]
    kps[KP_WRIST_L]    = [wrist_x, wrist_y, conf]
    kps[KP_WRIST_R]    = [wrist_x+40, wrist_y, conf]
    return kps

print("═" * 55)
print("  TESTS UNITAIRES — RiskScorerV2 + Agent LangGraph")
print("═" * 55)

# TEST 1 — Personne neutre loin du chien : final_score H→D doit être faible
def t1():
    sc = RiskScorerV2(history_len=8)
    kps = _make_kps(shoulder_y=250, wrist_y=260, wrist_x=320)  # Bras le long du corps
    p_bbox = np.array([280.0, 200.0, 360.0, 400.0])
    d_bbox = np.array([700.0, 300.0, 800.0, 380.0])            # Chien à 400px+
    rd = sc.compute_pair(1, kps, p_bbox, 1, d_bbox)
    score = rd['human_to_dog']['final_score']
    assert score < 0.20, f"Score attendu < 0.20, obtenu {score:.3f}"
_test("TEST 1 — Personne neutre loin du chien (H→D < 0.20)", t1)

# TEST 2 — Bras levé orienté vers le chien : final_score H→D doit être élevé
def t2():
    sc = RiskScorerV2(history_len=8)
    p_bbox = np.array([280.0, 200.0, 360.0, 400.0])
    d_bbox = np.array([420.0, 290.0, 490.0, 360.0])            # Chien proche (30px bord)
    kps_prev = _make_kps(shoulder_y=250, wrist_y=240, wrist_x=300)  # Position initiale
    sc.compute_pair(2, kps_prev, p_bbox, 2, d_bbox)             # Initialise historique
    kps_curr = _make_kps(shoulder_y=250, wrist_y=170, wrist_x=390)  # Poignet levé vers chien
    rd = sc.compute_pair(2, kps_curr, p_bbox, 2, d_bbox)
    score = rd['human_to_dog']['final_score']
    assert score > 0.30, f"Score attendu > 0.30, obtenu {score:.3f}"
_test("TEST 2 — Bras levé orienté vers le chien (H→D > 0.30)", t2)

# TEST 3 — Bras levé mais orienté ailleurs : wrist_toward_dog annule le score
def t3():
    sc = RiskScorerV2(history_len=8)
    p_bbox = np.array([280.0, 200.0, 360.0, 400.0])
    d_bbox = np.array([420.0, 290.0, 490.0, 360.0])
    kps_prev = _make_kps(shoulder_y=250, wrist_y=240, wrist_x=300)
    sc.compute_pair(3, kps_prev, p_bbox, 3, d_bbox)
    # Poignet levé mais se déplace à gauche (loin du chien qui est à droite)
    kps_curr = _make_kps(shoulder_y=250, wrist_y=170, wrist_x=100)
    rd = sc.compute_pair(3, kps_curr, p_bbox, 3, d_bbox)
    wrist_dir = rd['human_to_dog']['details']['wrist_toward_dog']
    assert wrist_dir < 0.35, f"wrist_toward_dog attendu < 0.35, obtenu {wrist_dir:.3f}"
_test("TEST 3 — Bras levé pointant ailleurs (wrist_dir < 0.35)", t3)

# TEST 4 — Chien fonce vers humain : final_score D→H doit être élevé
def t4():
    sc = RiskScorerV2(history_len=8)
    p_bbox   = np.array([300.0, 250.0, 380.0, 420.0])
    d_bbox1  = np.array([360.0, 300.0, 420.0, 360.0])          # Chien à 20px du bord gauche
    sc.compute_pair(4, None, p_bbox, 4, d_bbox1)
    d_bbox2  = np.array([340.0, 280.0, 400.0, 340.0])          # Chien se rapproche
    rd = sc.compute_pair(4, None, p_bbox, 4, d_bbox2)
    score = rd['dog_to_human']['final_score']
    assert score > 0.15, f"Score D→H attendu > 0.15, obtenu {score:.3f}"
_test("TEST 4 — Chien se rapproche de l'humain (D→H > 0.15)", t4)

# TEST 5 — Chien rapide mais s'éloignant : dog_approach_score doit être faible
def t5():
    sc = RiskScorerV2(history_len=8)
    p_bbox  = np.array([300.0, 250.0, 380.0, 420.0])
    d_bbox1 = np.array([400.0, 290.0, 460.0, 350.0])
    sc.compute_pair(5, None, p_bbox, 5, d_bbox1)
    d_bbox2 = np.array([550.0, 290.0, 610.0, 350.0])           # Chien s'éloigne vite
    rd = sc.compute_pair(5, None, p_bbox, 5, d_bbox2)
    appr = rd['dog_to_human']['details']['dog_approach_score']
    assert appr < 0.15, f"dog_approach_score attendu < 0.15, obtenu {appr:.3f}"
_test("TEST 5 — Chien rapide mais s'éloignant (approach < 0.15)", t5)

# TEST 6 — Lissage EMA : alternance 0.8/0.1 ne dépasse pas le seuil
def t6():
    sc = RiskScorerV2(history_len=8)
    tk = ('h2d', 10, 10)
    # Simulation 10 frames alternées 0.8 / 0.1
    for i in range(10):
        sc._update_ema(tk, 0.8 if i % 2 == 0 else 0.1)
    ema_alt = sc.ema_scores[tk]
    assert ema_alt < RISK_THRESHOLD, (
        f"EMA alternée {ema_alt:.3f} devrait être < seuil {RISK_THRESHOLD}"
    )
    # Simulation 10 frames à 0.8 constant → convergence
    sc2 = RiskScorerV2(history_len=8)
    for _ in range(10):
        sc2._update_ema(('h2d',20,20), 0.8)
    ema_const = sc2.ema_scores[('h2d',20,20)]
    assert ema_const > 0.7, f"EMA constante {ema_const:.3f} devrait converger vers 0.8"
_test("TEST 6 — Lissage EMA (alternée < seuil, constante > 0.70)", t6)

# TEST 7 — Agent routing minor → stocker seulement
def t7():
    if not LANGGRAPH_OK or agent_graph is None:
        raise AssertionError("LangGraph non disponible — skip")
    mock_state = {
        'incident_id': 'test-minor-001', 'timestamp': datetime.now().isoformat(),
        'video_source': 'test', 'frame_number': 1,
        'incident_type': 'human_to_dog', 'person_track_id': 1, 'dog_track_id': 1,
        'distance_px': 80.0, 'risk_scores': {}, 'ema_score': 0.55,
        'ema_history': [0.55], 'sustained_frames': 5, 'evidence_list': [],
        'llm_confirmed': True, 'llm_confidence': 0.75, 'llm_reason': 'test',
        'llm_severity': 'minor', 'llm_recommended_action': 'report_only',
        'llm_model': LLM_MODEL, 'severity': '', 'db_event_id': -1,
        'report_path': '', 'capture_path': '', 'annotated_frame': b'',
        'notifications_sent': [], 'actions_taken': [], 'routing_decision': '',
        'processing_time_ms': 0.0
    }
    result = agent_graph.invoke(mock_state)
    assert result['routing_decision'] == 'stocker', (
        f"routing_decision attendu='stocker', obtenu='{result['routing_decision']}'"
    )
    assert 'email_sent' not in result.get('actions_taken', []), (
        "email_sent ne devrait pas être dans les actions pour severity=minor"
    )
_test("TEST 7 — Agent routing minor → stocker (pas d'email)", t7)

# TEST 8 — Agent routing critical dog_to_human → email + urgences
def t8():
    if not LANGGRAPH_OK or agent_graph is None:
        raise AssertionError("LangGraph non disponible — skip")
    mock_state = {
        'incident_id': 'test-critical-001', 'timestamp': datetime.now().isoformat(),
        'video_source': 'test', 'frame_number': 42,
        'incident_type': 'dog_to_human', 'person_track_id': 2, 'dog_track_id': 3,
        'distance_px': 10.0, 'risk_scores': {}, 'ema_score': 0.88,
        'ema_history': [0.85, 0.88], 'sustained_frames': 12,
        'evidence_list': ['chien_fonce:0.80', 'personne_fuit:0.65'],
        'llm_confirmed': True, 'llm_confidence': 0.92, 'llm_reason': 'test critical',
        'llm_severity': 'critical', 'llm_recommended_action': 'notify_medical',
        'llm_model': LLM_MODEL, 'severity': '', 'db_event_id': -1,
        'report_path': '', 'capture_path': '', 'annotated_frame': b'',
        'notifications_sent': [], 'actions_taken': [], 'routing_decision': '',
        'processing_time_ms': 0.0
    }
    result = agent_graph.invoke(mock_state)
    assert result['routing_decision'] == 'urgences', (
        f"routing attendu='urgences', obtenu='{result['routing_decision']}'"
    )
    assert 'incident_stored' in result.get('actions_taken', []), (
        "incident_stored devrait être dans actions_taken"
    )
    assert 'report_generated' in result.get('actions_taken', []), (
        "report_generated devrait être dans actions_taken"
    )
_test("TEST 8 — Agent routing critical → urgences + stockage + rapport", t8)

print(f"\n{'═'*55}")
print(f"  Résultats : {_pass} ✅ réussis  |  {_fail} ❌ échoués")
print(f"{'═'*55}")

---
## 🔬 Évaluation — Différenciation Humain→Chien vs Chien→Humain

Cette section analyse si notre système peut réellement distinguer les deux scénarios d'agression.

### Problème identifié
Les scores H→D et D→H peuvent se **chevaucher** dans plusieurs cas :
- Un humain qui recule devant un chien qui attaque déclenche des signaux H→D (mouvement de bras)
- Un chien calme près d'un humain agité peut déclencher des signaux D→H (proximité + vitesse)
- La proximity_factor est identique pour les deux scores → elle ne différencie pas la *direction*

### Méthode d'évaluation
On génère des scénarios synthétiques avec des comportements clairement H→D ou D→H, 
on passe les scores dans RiskScorerV2, et on mesure :
- **TPR** (True Positive Rate) par type
- **Taux de confusion** H→D classé comme D→H et vice versa
- **Score de séparabilité** = (écart moyen entre scores) / (somme des écarts-types)


In [ ]:
# ─── ÉVALUATION A : Scénarios synthétiques ───────────────────────────────────
# Génère des paires (keypoints, bbox) avec des comportements clairement H→D ou D→H
# et mesure si RiskScorerV2 les classe correctement.

import numpy as np
from collections import defaultdict, deque

# ── Helpers de génération de scénarios ───────────────────────────────────────

def _make_kps_h2d_attack(wrist_toward_dog=True, arm_raised=True, ankle_near_dog=False):
    """Simule un humain agressif vers un chien : bras levé, poignet orienté vers le chien."""
    kps = np.zeros((17, 3))
    kps[:, 2] = 0.9  # High confidence for all keypoints
    # Épaules
    kps[5] = [300, 250, 0.9]   # Left shoulder
    kps[6] = [340, 250, 0.9]   # Right shoulder
    # Coudes levés si arm_raised
    elbow_y = 200 if arm_raised else 270
    kps[7] = [300, elbow_y, 0.9]  # Left elbow
    kps[8] = [340, elbow_y, 0.9]  # Right elbow
    # Poignets orientés vers le chien (droite, x=420) si wrist_toward_dog
    wrist_x = 390 if wrist_toward_dog else 260  # toward or away from dog
    wrist_y = 170 if arm_raised else 265
    kps[9]  = [wrist_x - 10, wrist_y, 0.9]
    kps[10] = [wrist_x, wrist_y, 0.9]
    # Hanches
    kps[11] = [305, 310, 0.9]
    kps[12] = [335, 310, 0.9]
    # Genoux
    kps[13] = [305, 370, 0.9]
    kps[14] = [335, 370, 0.9]
    # Chevilles (proches du chien si kick)
    ankle_x = 410 if ankle_near_dog else 310
    kps[15] = [ankle_x - 10, 430, 0.9]
    kps[16] = [ankle_x, 430, 0.9]
    return kps

def _make_kps_d2h_victim(person_retreating=True):
    """Simule un humain reculant devant un chien : bras en défense, position neutre."""
    kps = np.zeros((17, 3))
    kps[:, 2] = 0.9
    kps[5] = [300, 250, 0.9]; kps[6] = [340, 250, 0.9]   # Shoulders
    kps[7] = [290, 280, 0.9]; kps[8] = [350, 280, 0.9]   # Elbows low (arms down)
    # Poignets en défense (bras croisés ou bras vers l'avant)
    kps[9]  = [285, 300, 0.9]; kps[10] = [355, 300, 0.9]
    kps[11] = [305, 310, 0.9]; kps[12] = [335, 310, 0.9]
    kps[13] = [305, 370, 0.9]; kps[14] = [335, 370, 0.9]
    kps[15] = [300, 430, 0.9]; kps[16] = [340, 430, 0.9]
    return kps

# ── Scénarios de test ─────────────────────────────────────────────────────────

FRAME_W, FRAME_H = 1280, 720
FRAME_DIAG = np.hypot(FRAME_W, FRAME_H)

# Person bbox (static)
P_BBOX = np.array([270.0, 200.0, 370.0, 440.0])
# Dog bbox: proche (edge-to-edge ~50px)
D_BBOX_CLOSE = np.array([380.0, 270.0, 460.0, 380.0])
# Dog bbox: moyen (edge-to-edge ~200px)
D_BBOX_FAR = np.array([600.0, 280.0, 680.0, 370.0])

scenarios_h2d = [
    {"name": "H→D: Bras levé + poignet vers chien (proche)",  "kps_fn": lambda: _make_kps_h2d_attack(True,  True,  False), "d_bbox": D_BBOX_CLOSE},
    {"name": "H→D: Coup de pied vers chien (proche)",          "kps_fn": lambda: _make_kps_h2d_attack(True,  True,  True),  "d_bbox": D_BBOX_CLOSE},
    {"name": "H→D: Bras levé mais chien loin",                "kps_fn": lambda: _make_kps_h2d_attack(True,  True,  False), "d_bbox": D_BBOX_FAR},
    {"name": "H→D: Poignet orienté ailleurs (faux positif?)", "kps_fn": lambda: _make_kps_h2d_attack(False, True,  False), "d_bbox": D_BBOX_CLOSE},
]

scenarios_d2h = [
    {"name": "D→H: Chien rapide vers humain (proche)",  "kps_fn": _make_kps_d2h_victim, "d_bbox": D_BBOX_CLOSE, "dog_speed": 0.15},
    {"name": "D→H: Chien rapide vers humain (moyen)",   "kps_fn": _make_kps_d2h_victim, "d_bbox": D_BBOX_FAR,   "dog_speed": 0.15},
    {"name": "D→H: Chien lent, direction cohérente",    "kps_fn": _make_kps_d2h_victim, "d_bbox": D_BBOX_CLOSE, "dog_speed": 0.04},
]

print(f"{'Scénario':<50} {'H→D':>6} {'D→H':>6} {'Classé':>10} {'Correct':>8}")
print("─" * 85)

h2d_correct = 0; d2h_correct = 0; h2d_total = len(scenarios_h2d); d2h_total = len(scenarios_d2h)

for sc in scenarios_h2d:
    scorer = RiskScorerV2(history_len=8)
    kps_prev = _make_kps_h2d_attack(False, False, False)
    scorer.compute_pair(1, kps_prev, P_BBOX, 1, D_BBOX_CLOSE)  # warm up history
    kps = sc["kps_fn"]()
    rd = scorer.compute_pair(1, kps, P_BBOX, 1, sc["d_bbox"])
    h2d = rd["human_to_dog"]["final_score"]
    d2h = rd["dog_to_human"]["final_score"]
    classified = "H→D" if h2d >= d2h else "D→H"
    correct = "✅" if classified == "H→D" else "❌"
    if classified == "H→D": h2d_correct += 1
    print(f"{sc['name']:<50} {h2d:>6.3f} {d2h:>6.3f} {classified:>10} {correct:>8}")

print()
for sc in scenarios_d2h:
    scorer = RiskScorerV2(history_len=8)
    # Simulate dog movement history toward person
    dog_speed = sc.get("dog_speed", 0.10)
    d_bbox_start = sc["d_bbox"].copy() + np.array([dog_speed*FRAME_W*3, 0, dog_speed*FRAME_W*3, 0])
    for step in range(5):
        d_bbox_t = d_bbox_start - np.array([step*dog_speed*FRAME_W, 0, step*dog_speed*FRAME_W, 0])
        kps = sc["kps_fn"]()
        rd = scorer.compute_pair(1, kps, P_BBOX, 1, d_bbox_t)
    h2d = rd["human_to_dog"]["final_score"]
    d2h = rd["dog_to_human"]["final_score"]
    classified = "D→H" if d2h >= h2d else "H→D"
    correct = "✅" if classified == "D→H" else "❌"
    if classified == "D→H": d2h_correct += 1
    print(f"{sc['name']:<50} {h2d:>6.3f} {d2h:>6.3f} {classified:>10} {correct:>8}")

print(f"\n{'─'*85}")
print(f"Précision H→D : {h2d_correct}/{h2d_total} = {h2d_correct/h2d_total:.0%}")
print(f"Précision D→H : {d2h_correct}/{d2h_total} = {d2h_correct/d2h_total:.0%}")


---
## 📊 Analyse et améliorations proposées

### Problèmes identifiés par l'évaluation

1. **Chevauchement des scores dans la zone proche** : Quand le chien est très proche (<50px),
   la `proximity_factor` booste les DEUX scores — le système ne peut pas distinguer qui initie.

2. **Score D→H trop dépendant de la vitesse brute** : Un chien joueur ou excité déclenche
   le même signal D→H qu'un chien agressif.

3. **Pas de prise en compte du recul humain** : Si la personne recule (signe qu'elle est victime),
   cela ne réduit pas le score H→D.

4. **incident_type assigné au moment de la confirmation** : Pendant la phase ACTIVE, le type
   peut changer à chaque frame. On prend la valeur courante au moment du trigger, pas la majorité.

### Améliorations implémentées dans le backend (`video_analyzer.py`)

| Amélioration | Description | Impact |
|---|---|---|
| **Vote temporel** | Compter les votes H→D vs D→H sur toute la phase ACTIVE, assigner la majorité | Réduit les bascules de dernière minute |
| **Retraite humaine** | Détecter si la personne recule (CoG se déplace à l'opposé du chien) → réduire H→D | Distingue victime vs agresseur |
| **Ratio de confiance** | Ne labeler que si score dominant > 1.5× l'autre, sinon "ambiguous" | Évite les faux labels |
| **Heading chien** | Utiliser l'orientation du chien (déjà calculée) pour booster D→H | Meilleure détection des charges |


In [ ]:
# ─── ÉVALUATION B : Impact des améliorations ─────────────────────────────────
# Compare le comportement avant/après les corrections proposées.

def improved_incident_type(h2d_scores: list, d2h_scores: list,
                           confidence_ratio: float = 1.5) -> str:
    """
    Determine incident type using majority vote + confidence ratio.
    
    Args:
        h2d_scores: List of h2d scores during the ACTIVE phase
        d2h_scores: List of d2h scores during the ACTIVE phase
        confidence_ratio: Minimum ratio of dominant/other score to label confidently
    Returns:
        "human_to_dog" | "dog_to_human" | "ambiguous"
    """
    if not h2d_scores or not d2h_scores:
        return "unknown"
    
    h2d_mean = np.mean(h2d_scores)
    d2h_mean = np.mean(d2h_scores)
    
    if h2d_mean > d2h_mean:
        ratio = h2d_mean / max(d2h_mean, 1e-6)
        return "human_to_dog" if ratio >= confidence_ratio else "ambiguous"
    else:
        ratio = d2h_mean / max(h2d_mean, 1e-6)
        return "dog_to_human" if ratio >= confidence_ratio else "ambiguous"


def person_retreating(person_centers: list, dog_bbox: np.ndarray) -> bool:
    """
    Check if the person is moving AWAY from the dog over the last N frames.
    Returns True if the person is retreating (sign they're a victim, not aggressor).
    """
    if len(person_centers) < 4:
        return False
    
    # Dog center
    dog_cx = (dog_bbox[0] + dog_bbox[2]) / 2
    dog_cy = (dog_bbox[1] + dog_bbox[3]) / 2
    
    # Distance from person to dog at start and end of window
    px_start, py_start = person_centers[-4]
    px_end,   py_end   = person_centers[-1]
    
    dist_start = np.hypot(px_start - dog_cx, py_start - dog_cy)
    dist_end   = np.hypot(px_end   - dog_cx, py_end   - dog_cy)
    
    return dist_end > dist_start * 1.1  # Person moved at least 10% farther from dog

# Simulate a sequence where person retreats and dog advances
print("Test: personne qui recule devant un chien qui avance")
print("─" * 60)

h2d_scores_list = []
d2h_scores_list = []
scorer_improved = RiskScorerV2(history_len=8)
person_positions = []
dog_positions_advancing = []

# Simulate 8 frames: dog advances from x=500 to x=390, person retreats from x=270 to x=220
for frame in range(8):
    dog_x_offset = -14 * frame   # Dog moves left (toward person)
    person_x_offset = -6 * frame # Person moves left (retreating)
    
    d_bbox = np.array([500 + dog_x_offset, 280, 580 + dog_x_offset, 370], dtype=float)
    p_bbox = np.array([270 + person_x_offset, 200, 370 + person_x_offset, 440], dtype=float)
    kps = _make_kps_d2h_victim()
    kps[:, 0] += person_x_offset  # Shift keypoints with person
    
    rd = scorer_improved.compute_pair(1, kps, p_bbox, 1, d_bbox)
    h2d_scores_list.append(rd["human_to_dog"]["final_score"])
    d2h_scores_list.append(rd["dog_to_human"]["final_score"])
    person_cx = (p_bbox[0]+p_bbox[2])/2
    person_cy = (p_bbox[1]+p_bbox[3])/2
    person_positions.append((person_cx, person_cy))

# Final dog bbox
final_d_bbox = np.array([500 + (-14*7), 280, 580 + (-14*7), 370], dtype=float)

# Old method: take current values
old_h2d = h2d_scores_list[-1]
old_d2h = d2h_scores_list[-1]
old_type = "human_to_dog" if old_h2d >= old_d2h else "dog_to_human"

# New method: majority vote + confidence ratio + retreat detection
new_type = improved_incident_type(h2d_scores_list, d2h_scores_list)
retreating = person_retreating(person_positions, final_d_bbox)
if retreating and new_type == "human_to_dog":
    new_type = "dog_to_human"  # Override: person is retreating → they're the victim

print(f"Scores moyens  — H→D: {np.mean(h2d_scores_list):.3f}  D→H: {np.mean(d2h_scores_list):.3f}")
print(f"Méthode actuelle  : {old_type} (basée sur dernière frame seulement)")
print(f"Méthode améliorée : {new_type} (vote temporel + détection recul)")
print(f"Personne recule   : {'✅ Oui' if retreating else '❌ Non'}")
print(f"Résultat correct  : {'✅' if new_type == 'dog_to_human' else '❌'} (attendu: dog_to_human)")


---
## 🛠️ Recommandations finales

### Améliorations déjà intégrées dans le backend (`video_analyzer.py` v2)
Ces améliorations ont été intégrées dans `pet-advisor/backend/aggression/video_analyzer.py` :
- ✅ **Vote temporel** : `_TrackState` accumule `h2d_votes` et `d2h_votes` à chaque frame ACTIVE
- ✅ **Ratio de confiance** : `incident_type = "ambiguous"` si ratio < 1.5
- ✅ **Détection de recul** : réduction du score H→D si personne s'éloigne du chien
- ✅ **Heading chien** : déjà implémenté dans `_dog_heading()`, maintenant utilisé pour booster D→H

### Limitations inhérentes au système
1. **Pas de keypoints chien** : YOLOv8-pose ne fournit des keypoints que pour les humains
2. **Ambiguïté proximité** : quand les deux sujets sont très proches, les scores se chevauchent systématiquement
3. **Résolution** : sur caméras basse résolution, les keypoints manquent de précision

### Ce qu'il faudrait pour aller plus loin
- Un modèle de classification fine-tuné sur des clips étiquetés H→D / D→H
- Des keypoints canins (modèles spécialisés : AP-10K, AnimalPose)
- Un historique sonore (grognements → D→H, cris humains → D→H victime)


In [ ]:
# ─── CELLULE 17 : Export et téléchargement ───────────────────────────────────
# Exporte les incidents en CSV et télécharge les fichiers de sortie.

import zipfile
from google.colab import files

# Export CSV de tous les incidents
csv_path = "/content/incidents_export.csv"
export_to_csv(csv_path)
print(f"✅ CSV exporté : {csv_path}")

# Création d'une archive ZIP des captures + rapports
zip_path = "/content/aggression_results.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Vidéo annotée
    out_mp4 = "/content/output_annotated.mp4"
    if os.path.exists(out_mp4):
        zf.write(out_mp4, "output_annotated.mp4")

    # Base de données SQLite
    if os.path.exists(DB_PATH):
        zf.write(DB_PATH, "aggression_events.db")

    # CSV incidents
    if os.path.exists(csv_path):
        zf.write(csv_path, "incidents_export.csv")

    # Captures JPEG
    for fname in os.listdir(CAPTURES_DIR):
        fpath = os.path.join(CAPTURES_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, f"captures/{fname}")

    # Rapports texte
    if os.path.exists(REPORTS_DIR):
        for fname in os.listdir(REPORTS_DIR):
            fpath = os.path.join(REPORTS_DIR, fname)
            if os.path.isfile(fpath):
                zf.write(fpath, f"reports/{fname}")

size_mb = os.path.getsize(zip_path) / 1e6
print(f"✅ Archive ZIP : {zip_path} ({size_mb:.1f} MB)")

# Téléchargement depuis Colab
files.download(zip_path)
print("✅ Téléchargement lancé")

---
## 🔌 Cellule 18 — Export config pour le backend (`aggression_config.json`)

Le backend FastAPI (`backend/aggression/tools.py`) lit ce fichier au démarrage
pour résoudre les **destinataires** (hôpital / association droits des animaux)
et les **seuils de sévérité**. Cette cellule est **idempotente** : on peut la
ré-exécuter à tout moment pour mettre à jour la config sans toucher au code.

> **À renseigner avant de lancer en production** : les emails et libellés
> ci-dessous. Les valeurs par défaut sont des placeholders de démonstration.


In [ ]:
import requests,json,base64

BACKEND_URL='http://localhost:8000'

def forward_incident(incident_id):
    '''
    Maps notebook report fields -> AggressionIncidentRequest backend schema.
    Sends image as base64 in JSON body (not multipart).
    '''
    rep=REPORT_DIR/f'{incident_id}_report.json'
    mosaic=KEYFRAME_DIR/f'{incident_id}_mosaic.jpg'
    if not rep.exists():print(f'Missing: {rep}');return {}
    data=json.loads(rep.read_text());llm=data.get('llm_result',{})
    img=''
    if mosaic.exists():img=base64.b64encode(mosaic.read_bytes()).decode()
    payload={
        'incident_id':data['incident_id'],
        'timestamp':data['timestamp_iso'],
        'video_source':data.get('video_source',''),
        'person_track_id':data.get('person_track_id',-1),
        'dog_track_id':data.get('dog_track_id',-1),
        'frame_number':data.get('frame_number',-1),
        'distance_px':data.get('distance_px',0.),
        'ema_score':data.get('ema_risk_at_trigger',0.),
        'sustained_frames':data.get('sustained_frames',0),
        'incident_type':llm.get('aggression_type','unknown'),
        'evidence_list':llm.get('key_observations',[]),
        'llm_confirmed':llm.get('is_aggression',False),
        'llm_confidence':llm.get('confidence',0.),
        'llm_severity_hint':llm.get('severity','none'),
        'llm_reason':llm.get('report_markdown','')[:500],
        'llm_recommended_action':llm.get('recommended_action',''),
        'location_label':data.get('location_label',''),
        'image_jpeg_b64':img,
    }
    try:
        r=requests.post(f'{BACKEND_URL}/aggression/incident',json=payload,timeout=30)
        r.raise_for_status();res=r.json()
        print(f'OK {incident_id} db_id={res.get("db_event_id","?")}');return res
    except Exception as e:print(f'Failed:{e}');return {'error':str(e)}

def forward_all_incidents():
    reps=sorted(REPORT_DIR.glob('*_report.json'))
    if not reps:print('No reports');return
    for p in reps:forward_incident(p.stem.replace('_report',''))

print(f'Forwarder ready | {BACKEND_URL} | {REPORT_DIR}')


---
## 📡 Cellule 19 — Forwarder les incidents vers le backend (optionnel)

Quand `BACKEND_URL` est défini (cellule 04), chaque incident confirmé par le
LLM est **dupliqué** vers l'API FastAPI : c'est le backend qui prend alors le
relais pour la décision (sévérité re-évaluée, choix des canaux à contacter,
génération du rapport, persistance SQLite).

Ce mode est utile pour la **production** : la détection tourne sur un GPU
Colab/edge, mais la décision et la traçabilité vivent dans le serveur web qui
sert l'application d'adoption.

> **Mode mixte autorisé** : on peut laisser l'agent LangGraph local tourner
> *en plus* de l'envoi backend — les deux sont indépendants. L'incident finira
> dans deux SQLite distincts, ce qui permet la réconciliation après coup.


In [ ]:
# ─── CELLULE 19 : Forwarder vers le backend (optionnel) ──────────────────────
# Branche-toi sur l'agent LangGraph existant : si BACKEND_URL est défini, on
# pose chaque incident confirmé sur l'endpoint POST /aggression/incident.
# Le backend re-évalue la sévérité, décide des canaux, écrit le rapport et
# stocke en DB. Aucune dépendance lourde — uniquement requests.

import requests, base64

def forward_incident_to_backend(state: dict, annotated_jpeg: bytes = b"") -> dict:
    """Best-effort POST: returns the backend response or an error dict.
    The detector loop must NOT crash if the backend is unreachable — we only
    log and continue. The local LangGraph agent (cell 09) is the redundant path.
    """
    if not BACKEND_URL:
        return {"status": "skipped", "reason": "BACKEND_URL not set"}

    # Map detector state -> backend AggressionIncidentRequest schema.
    payload = {
        "incident_id":       state.get("incident_id", ""),
        "timestamp":         state.get("timestamp", ""),
        "video_source":      str(state.get("video_source", "")),
        "frame_number":      int(state.get("frame_number", -1)),

        "incident_type":     state.get("incident_type", "unknown"),
        "person_track_id":   int(state.get("person_track_id", -1)),
        "dog_track_id":      int(state.get("dog_track_id", -1)),
        "distance_px":       float(state.get("distance_px", 0.0)),

        "ema_score":         float(state.get("ema_score", 0.0)),
        "sustained_frames":  int(state.get("sustained_frames", 0)),
        "evidence_list":     list(state.get("evidence_list", [])),

        "llm_confirmed":          bool(state.get("llm_confirmed", False)),
        "llm_confidence":         float(state.get("llm_confidence", 0.0)),
        "llm_severity_hint":      state.get("llm_severity", "minor"),
        "llm_reason":             state.get("llm_reason", ""),
        "llm_recommended_action": state.get("llm_recommended_action", ""),

        "location_label":  os.environ.get("AGGRESSION_LOCATION_LABEL", ""),
        "image_jpeg_b64":  base64.b64encode(annotated_jpeg).decode("ascii") if annotated_jpeg else "",
    }
    try:
        resp = requests.post(
            f"{BACKEND_URL.rstrip('/')}/aggression/incident",
            json=payload,
            timeout=8.0,
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as exc:
        logger.warning("forward_incident_to_backend failed: %s", exc)
        return {"status": "error", "error": str(exc)}


print("✅ forward_incident_to_backend() prête.")
print(f"   BACKEND_URL = {BACKEND_URL or '(non défini — mode local seul)'}")
print("   Pour activer : os.environ['PETADVISOR_BACKEND_URL'] = 'http://localhost:8000'")
print("   Puis brancher dans run_pipeline() à la place ou en plus de l'agent local.")


---

## Guide de personnalisation

### Ajuster la sensibilité

| Paramètre | Valeur par défaut | Effet |
|-----------|-----------------|-------|
| `RISK_THRESHOLD` | `0.50` | ↑ = moins de faux positifs, ↓ = plus sensible |
| `EMA_ALPHA` | `0.35` | ↑ = plus réactif aux changements rapides |
| `MIN_SUSTAINED_FRAMES` | `4` | ↑ = exige plus de frames avant LLM |
| `PROXIMITY_THRESHOLD_PX` | `120` | Calibrer selon la résolution de la caméra |
| `MIN_FRAMES_BETWEEN_LLM` | `375` | ≈15s à 25fps — anti-spam LLM |

### Activer les notifications email

1. Dans Colab → **Secrets** (🔑), ajouter :
   - `GEMINI_API_KEY` : votre clé API Google AI
   - `SMTP_USER` : adresse Gmail expéditrice
   - `SMTP_PASS` : mot de passe d'application Google
2. Dans la **cellule 04** : `EMAIL_ENABLED = True`
3. Renseigner `ALERT_RECIPIENTS`, `SPA_EMAIL`, `MEDICAL_EMAIL`

### Utiliser votre propre vidéo

Dans la **cellule 13** : changer `SOURCE_TYPE` en `"drive"`, `"youtube"` ou `"upload"`.

### Accélérer sur Colab Pro

- Runtime → Changer le type de runtime → **GPU A100** (Colab Pro+)
- `USE_FP16 = True` (activé automatiquement si GPU détecté)
- Réduire `MAX_FRAMES` dans la cellule 14 pour des tests rapides

### Schéma SQLite v2

La base `aggression_events.db` contient toutes les colonnes de `IncidentState`.
Requêtes utiles :

```python
# Incidents critiques
get_incidents_by_severity('critical', limit=10)

# Incidents chien→humain
get_incidents_by_type('dog_to_human', limit=20)

# Stats globales
get_stats_summary()

# Export CSV
export_to_csv('/content/my_export.csv')
```

### Architecture multi-caméras (avancé)

Pour gérer plusieurs flux en parallèle, instancier un `LLMWorker` et un
`RiskScorerV2` par flux dans des threads séparés. L'`agent_graph` est
thread-safe une fois compilé.